# LeetCode 214 — Shortest Palindrome (Hard) bằng Python

Notebook này giảng **từ căn bản** một bài chuỗi kinh điển: biến một chuỗi thành **palindrome ngắn nhất** bằng cách chỉ được **thêm ký tự vào phía trước**. Skeleton trên LeetCode (Python3):

```python
class Solution:
    def shortestPalindrome(self, s: str) -> str:

```

---

## Sau notebook này, bạn sẽ nắm được

1. Cách **quy bài toán** về một câu hỏi gọn hơn nhiều: *tiền tố palindrome dài nhất* (longest palindromic prefix) của `s` dài bao nhiêu? — và chứng minh vì sao được phép quy như vậy.
2. **Brute force**: thử tiền tố từ dài đến ngắn, mẹo so `s[:i]` với `rev[n-i:]`, rồi **đếm phép so sánh** để tận mắt thấy `O(n²)`.
3. **Hai con trỏ + đệ quy** (cách trong editorial của LeetCode): vì sao nó đúng, khi nào nó chậm, vì sao Python có thể báo `RecursionError` và cách khử đệ quy.
4. **KMP / mảng `lps` (prefix function)** học từ con số 0: tiền tố, hậu tố, "biên" (border), cách tính `lps` trong `O(n)` có trace từng bước — rồi dùng mẹo `s + "#" + reverse(s)` cho **lời giải chính `O(n)`**.
5. **Rolling hash** (băm cuộn, ý tưởng Rabin–Karp) như một cách `O(n)` khác, kèm rủi ro **va chạm hash** và cách tự kiểm tra lại cho chắc.
6. Lỗi hay gặp (quên `"#"`, `if` thay cho `while`, ghép chuỗi sai phía...), mẹo nhận dạng pattern, bài liên quan và **6 bài tập có đáp án**.

Lộ trình: `Thử mọi chuỗi X (hàm mũ)` → `Brute force tiền tố O(n²)` → `Hai con trỏ + đệ quy (xấu nhất O(n²))` → `KMP O(n)` (+ `Rolling hash O(n) kỳ vọng`).

## 1. Đề bài

Cho một chuỗi `s`. Bạn có thể biến `s` thành một **palindrome** (chuỗi đối xứng — đọc xuôi hay đọc ngược đều giống nhau) bằng cách **thêm ký tự vào phía trước** nó. Hãy trả về **palindrome ngắn nhất** có thể tạo ra bằng phép biến đổi này.

> Gốc tiếng Anh: *You are given a string `s`. You can convert `s` to a palindrome by adding characters in front of it. Return the shortest palindrome you can find by performing this transformation.*

### Ví dụ trong đề

| | Input `s` | Output |
|---|---|---|
| Ví dụ 1 | `"aacecaaa"` | `"aaacecaaa"` |
| Ví dụ 2 | `"abcd"` | `"dcbabcd"` |

### Constraints (ràng buộc)

- `0 <= s.length <= 5 * 10^4`
- `s` chỉ gồm chữ cái tiếng Anh viết thường (`'a'`–`'z'`).

Hai điều cần để ý ngay từ đầu:

- `n` có thể tới **50 000**. Thuật toán `O(n²)` về lý thuyết là cỡ `n² = 2.5 tỉ` bước — rất rủi ro. Mục tiêu của ta là `O(n)`.
- `s.length` có thể bằng **0** — chuỗi rỗng là input hợp lệ, đáp án khi đó là `""`.

## 2. Hiểu đúng đề: "chỉ được thêm vào phía trước"

"Thêm vào phía trước" nghĩa là kết quả luôn có dạng **`X + s`**, với `X` là chuỗi ta tự chọn:

- `s` phải nằm **nguyên vẹn ở cuối** kết quả. Không được chèn vào giữa, không được thêm vào cuối, không được xóa ký tự nào.
- Trong Python, `t.endswith(s)` kiểm tra chuỗi `t` có **kết thúc bằng** `s` hay không — đúng thứ ta cần để kiểm tra "dạng `X + s`".

Luôn có đáp án: chọn `X = reverse(s[1:])` thì `X + s` chắc chắn là palindrome (ví dụ `"dcb" + "abcd"`). Vậy kết quả **không bao giờ dài quá `2n - 1`**. Câu hỏi chỉ là: **ngắn nhất** được bao nhiêu?

Ví dụ với `s = "abcd"`:

| Ứng viên | Có dạng `X + s`? | Palindrome? | Hợp lệ? |
|---|:---:|:---:|---|
| `"dcbaabcd"` | ✔ (`X = "dcba"`) | ✔ | ✔ nhưng dài 8 |
| `"dcbabcd"` | ✔ (`X = "dcb"`) | ✔ | ✔ dài 7 ← **đáp án** |
| `"abcdcba"` | ✘ (đây là thêm vào **cuối**) | ✔ | ✘ |
| `"cbabcd"` | ✔ (`X = "cb"`) | ✘ | ✘ |

Ta viết 2 hàm nhỏ để dùng xuyên suốt notebook: `is_palindrome(t)` và `is_valid_answer(s, t)`.

In [1]:
def is_palindrome(t: str) -> bool:
    """Chuỗi t đọc xuôi và đọc ngược có giống nhau không?"""
    return t == t[::-1]


def is_valid_answer(s: str, t: str) -> bool:
    """t hợp lệ nếu: t là palindrome VÀ t có dạng X + s (tức s nằm ở cuối t)."""
    return is_palindrome(t) and t.endswith(s)


s = "abcd"
for cand in ["dcbaabcd", "dcbabcd", "abcdcba", "cbabcd"]:
    print(f"{cand:>9} | palindrome={is_palindrome(cand)!s:<5} | dạng X+s={cand.endswith(s)!s:<5} | hợp lệ={is_valid_answer(s, cand)}")

# Hai output mẫu của đề phải hợp lệ
assert is_valid_answer("aacecaaa", "aaacecaaa")
assert is_valid_answer("abcd", "dcbabcd")
print("Hai output mẫu của đề đều hợp lệ.")

 dcbaabcd | palindrome=True  | dạng X+s=True  | hợp lệ=True
  dcbabcd | palindrome=True  | dạng X+s=True  | hợp lệ=True
  abcdcba | palindrome=True  | dạng X+s=False | hợp lệ=False
   cbabcd | palindrome=False | dạng X+s=True  | hợp lệ=False
Hai output mẫu của đề đều hợp lệ.


## 3. Kiến thức nền 1 — Cắt chuỗi (slicing), tiền tố, hậu tố và "tiền tố palindrome"

Với `s = "abcd"` (độ dài `n = 4`), vị trí (index) đánh số từ `0`: `s[0] = 'a'`, ..., `s[3] = 'd'`.

| Cú pháp | Đọc là | Với `s = "abcd"` |
|---|---|---|
| `s[a:b]` | từ vị trí `a` đến **trước** `b` | `s[1:3] = "bc"` |
| `s[:i]` | **tiền tố** (prefix) dài `i` — `i` ký tự đầu | `s[:2] = "ab"` |
| `s[i:]` | phần còn lại sau tiền tố | `s[2:] = "cd"` |
| `s[n-k:]` | **hậu tố** (suffix) dài `k` — `k` ký tự cuối | `s[2:] = "cd"` (k = 2) |
| `s[::-1]` | đảo ngược (bước nhảy `-1`) | `"dcba"` |

⚠️ Bẫy nhỏ: `s[-k:]` cũng lấy `k` ký tự cuối, **trừ khi `k = 0`**: `s[-0:]` chính là `s[0:]` = **cả chuỗi**, không phải chuỗi rỗng. Vì vậy notebook này luôn viết hậu tố là `s[n-k:]`.

### "Tiền tố palindrome"

Một **tiền tố palindrome** là tiền tố `s[:i]` mà bản thân nó là palindrome. Mọi chuỗi không rỗng đều có ít nhất một tiền tố palindrome: `s[:1]` (một ký tự luôn đối xứng).

Ví dụ `s = "aacecaaa"`:

| `i` | `s[:i]` | palindrome? |
|:-:|---|:-:|
| 8 | `aacecaaa` | ✘ |
| 7 | `aacecaa` | ✔ ← dài nhất |
| 6 | `aaceca` | ✘ |
| 5 | `aacec` | ✘ |
| 4 | `aace` | ✘ |
| 3 | `aac` | ✘ |
| 2 | `aa` | ✔ |
| 1 | `a` | ✔ |

Khái niệm này sẽ là **chìa khóa** của cả bài — mục 5 sẽ chứng minh vì sao.

In [2]:
s = "abcd"
n = len(s)
print("s[0], s[3]            :", s[0], s[3])
print("s[1:3]                :", s[1:3])
print("s[:2]  (tiền tố dài 2):", s[:2])
print("s[2:]  (phần còn lại) :", s[2:])
print("s[n-2:] (hậu tố dài 2):", s[n - 2:])
print("s[::-1] (đảo ngược)   :", s[::-1])
print("Bẫy: s[-0:] =", repr(s[-0:]), "(cả chuỗi!)   còn s[n-0:] =", repr(s[n - 0:]))
print()

s = "aacecaaa"
for i in range(len(s), 0, -1):
    prefix = s[:i]
    print(f"i={i}  s[:i]={prefix:<9} palindrome? {is_palindrome(prefix)}")

s[0], s[3]            : a d
s[1:3]                : bc
s[:2]  (tiền tố dài 2): ab
s[2:]  (phần còn lại) : cd
s[n-2:] (hậu tố dài 2): cd
s[::-1] (đảo ngược)   : dcba
Bẫy: s[-0:] = 'abcd' (cả chuỗi!)   còn s[n-0:] = ''

i=8  s[:i]=aacecaaa  palindrome? False
i=7  s[:i]=aacecaa   palindrome? True
i=6  s[:i]=aaceca    palindrome? False
i=5  s[:i]=aacec     palindrome? False
i=4  s[:i]=aace      palindrome? False
i=3  s[:i]=aac       palindrome? False
i=2  s[:i]=aa        palindrome? True
i=1  s[:i]=a         palindrome? True


## 4. Cách 0 — Cách người mới tự nghĩ ra: thử thêm 0, 1, 2, ... ký tự

Câu hỏi tự nhiên nhất: *"Phải thêm ít nhất bao nhiêu ký tự?"* Vậy cứ thử:

- `k = 0`: không thêm gì — `s` đã là palindrome chưa?
- `k = 1`: thử **mọi** chuỗi `X` dài 1.
- `k = 2`: thử **mọi** chuỗi `X` dài 2. ...

Vì `k` tăng dần, chuỗi `X + s` palindrome **đầu tiên** tìm được chắc chắn là **ngắn nhất**.

`X` gồm những chữ cái nào? Chỉ cần thử các chữ cái **có trong `s`**: với `s` không rỗng, đáp án dài không quá `2n - 1` nên `k < n`, và khi đó mỗi ký tự của `X` (nằm ở nửa trái) "soi gương" đúng vào một ký tự của `s` — nên nó phải là một chữ cái của `s`.

Công cụ Python mới:

- `itertools.product(alphabet, repeat=k)` sinh **mọi bộ `k` ký tự** lấy từ `alphabet`. Ví dụ `product("ab", repeat=2)` cho `('a','a'), ('a','b'), ('b','a'), ('b','b')`.
- `"".join(bo_ky_tu)` nối một bộ/list ký tự thành chuỗi: `"".join(('d','c','b')) = "dcb"`.
- `sorted(set(s))`: `set(s)` là **tập** (set) các chữ cái khác nhau của `s` (bỏ trùng), `sorted` xếp chúng theo thứ tự cố định: `sorted(set("abca")) = ['a', 'b', 'c']`.
- Trong f-string, `{x!r}` in giá trị **kèm dấu nháy** (dạng `repr`) — nhờ vậy nhìn thấy rõ chuỗi rỗng `''`.

Cái giá: số ứng viên là `σ^k` (`σ` = số chữ cái khác nhau) — **hàm mũ**. Với 26 chữ cái và `k = 10` là khoảng `1.4 * 10^14` ứng viên. Không thể dùng cho bài thật, nhưng rất tốt để **hiểu đề** và làm "oracle" (bộ kiểm tra tuyệt đối tin cậy) cho chuỗi ngắn.

In [3]:
from itertools import product

print(["".join(p) for p in product("ab", repeat=2)])   # mọi chuỗi dài 2 từ {a, b}


def shortest_palindrome_try_all_X(s: str):
    """Thử k = 0, 1, 2, ... và MỌI chuỗi X dài k. Trả về (đáp án, k, số ứng viên đã thử)."""
    alphabet = sorted(set(s))
    tried = 0
    for k in range(len(s) + 1):
        for letters in product(alphabet, repeat=k):
            X = "".join(letters)
            tried += 1
            if is_palindrome(X + s):
                return X + s, k, tried
    return s, 0, tried      # chỉ để an toàn; thực tế luôn return ở trong vòng lặp


for s in ["aacecaaa", "abcd", "aab", ""]:
    ans, k, tried = shortest_palindrome_try_all_X(s)
    print(f"s={s!r:<11} -> {ans!r:<12} thêm k={k}, đã thử {tried} ứng viên")

['aa', 'ab', 'ba', 'bb']
s='aacecaaa'  -> 'aaacecaaa'  thêm k=1, đã thử 2 ứng viên
s='abcd'      -> 'dcbabcd'    thêm k=3, đã thử 79 ứng viên
s='aab'       -> 'baab'       thêm k=1, đã thử 3 ứng viên
s=''          -> ''           thêm k=0, đã thử 1 ứng viên


## 5. Hai quan sát biến bài Hard thành câu hỏi về "tiền tố palindrome"

### Quan sát 1: `X` không cần đoán — nó bị "ép" bởi `k`

Nhìn output trên: với `"abcd"`, `X = "dcb"` — đúng bằng **đảo ngược của 3 ký tự cuối** `"bcd"`. Không phải trùng hợp:

- `t = X + s` là palindrome, nên đọc `t` từ phải sang trái vẫn ra `t`.
- Vậy `k` ký tự **đầu** của `t` (chính là `X`) phải bằng đảo ngược của `k` ký tự **cuối** của `t`.
- `k` ký tự cuối của `t` chính là `k` ký tự cuối của `s`, tức `s[n-k:]`.

⟹ **`X = reverse(s[n-k:])`**. Với mỗi `k` chỉ có **một** ứng viên duy nhất — tổng cộng `n + 1` ứng viên thay vì `σ^k`.

### Quan sát 2: phần còn lại ở giữa phải là palindrome

Viết `t = X + s[:n-k] + s[n-k:]`. **Bóc `k` ký tự ở mỗi đầu** của palindrome `t` (bỏ `X` bên trái, bỏ `s[n-k:]` bên phải): phần giữa `s[:n-k]` vẫn là palindrome — giống như `"racecar"` bóc mỗi đầu 1 ký tự còn `"aceca"`.

Ngược lại, nếu `P = s[:n-k]` là palindrome và `R = s[n-k:]` thì `reverse(R) + P + R` luôn là palindrome (đọc ngược: `R` đảo thành `reverse(R)`, `P` vẫn là `P`, `reverse(R)` đảo thành `R`).

### Kết luận — bài toán thật sự cần giải

> Thêm được `k` ký tự ⟺ tiền tố `s[:n-k]` là palindrome.
>
> Muốn `k` **nhỏ nhất** ⟺ muốn tiền tố palindrome **dài nhất**. Gọi độ dài đó là **`L`**, thì:
>
> **`answer = reverse(s[L:]) + s`** — thêm đúng `n - L` ký tự.

```
s      =             [ P = s[:L] ][ R = s[L:] ]
answer = [ rev(R) ]  [ P = s[:L] ][ R = s[L:] ]
```

- `"aacecaaa"`: `L = 7` (`P = "aacecaa"`), `R = "a"` → `"a" + "aacecaaa" = "aaacecaaa"` ✔
- `"abcd"`: `L = 1` (`P = "a"`), `R = "bcd"` → `"dcb" + "abcd" = "dcbabcd"` ✔

Mảnh `reverse(s[L:])` còn viết được là `rev[:n-L]` với `rev = s[::-1]`: đảo cả chuỗi thì phần đuôi `R` (dài `n - L`) chạy lên đầu. Từ giờ, **mọi cách giải chỉ khác nhau ở cách tìm `L`**.

In [4]:
def build_answer(s: str, L: int) -> str:
    """Biết độ dài L của tiền tố palindrome dài nhất -> ghép đáp án."""
    R = s[L:]                 # phần đuôi KHÔNG nằm trong tiền tố palindrome
    return R[::-1] + s        # soi gương R rồi đặt lên trước


for s, L in [("aacecaaa", 7), ("abcd", 1), ("aab", 2)]:
    n = len(s)
    P, R = s[:L], s[L:]
    ans = build_answer(s, L)
    print(f"s={s:<9} P={P:<8} R={R:<4} rev(R)={R[::-1]:<4} -> {ans:<10} hợp lệ={is_valid_answer(s, ans)}")
    assert R[::-1] == s[::-1][:n - L]      # hai cách viết cùng một mảnh

s=aacecaaa  P=aacecaa  R=a    rev(R)=a    -> aaacecaaa  hợp lệ=True
s=abcd      P=a        R=bcd  rev(R)=dcb  -> dcbabcd    hợp lệ=True
s=aab       P=aa       R=b    rev(R)=b    -> baab       hợp lệ=True


## 6. Cách 1 — Brute force: thử tiền tố từ dài đến ngắn

Giờ chỉ cần tìm `L`. Cách tự nhiên nhất: thử `i = n, n-1, ..., 1`; tiền tố `s[:i]` **đầu tiên** là palindrome chính là tiền tố palindrome **dài nhất** (vì ta đi từ dài đến ngắn) → dừng ngay.

- `i = 1` luôn thỏa (một ký tự luôn là palindrome) → với `s` không rỗng, vòng lặp chắc chắn dừng và `L >= 1`.
- `s = ""`: `range(0, 0, -1)` rỗng, vòng lặp không chạy → `L = 0`, đáp án `""`.

### Mẹo nhỏ: chỉ đảo chuỗi MỘT lần

Kiểm tra `s[:i] == s[:i][::-1]` phải tạo một chuỗi đảo mới cho mỗi `i`. Thay vào đó, tính `rev = s[::-1]` **một lần**. Đảo ngược của tiền tố `s[:i]` chính là **hậu tố dài `i` của `rev`**:

`s[:i][::-1] == rev[n-i:]`

Ví dụ `s = "abcd"`, `rev = "dcba"`: `s[:3] = "abc"`, đảo lại là `"cba"`, đúng bằng `rev[1:] = "cba"` ✔.

Vậy: **`s[:i]` là palindrome ⟺ `s[:i] == rev[n-i:]`** — "tiền tố của `s` bằng hậu tố của `rev`". Hãy nhớ câu này: nó sẽ dẫn thẳng tới ý tưởng KMP ở mục 10.

In [5]:
def longest_pal_prefix_brute(s: str) -> int:
    for i in range(len(s), 0, -1):          # i = n, n-1, ..., 1
        if is_palindrome(s[:i]):
            return i
    return 0                                # chỉ xảy ra khi s rỗng


def shortest_palindrome_brute(s: str) -> str:
    return build_answer(s, longest_pal_prefix_brute(s))


def shortest_palindrome_rev_compare(s: str) -> str:
    n, rev = len(s), s[::-1]
    for i in range(n, 0, -1):
        if s[:i] == rev[n - i:]:            # tiền tố của s  vs  hậu tố của rev
            return rev[:n - i] + s          # rev[:n-i] chính là đảo của s[i:]
    return s                                # s rỗng


s = "aacecaaa"; rev = s[::-1]; n = len(s)
print("rev =", rev)
for i in range(n, 0, -1):
    print(f"i={i}: s[:i]={s[:i]:<9} rev[n-i:]={rev[n - i:]:<9} bằng nhau? {s[:i] == rev[n - i:]}")
    if s[:i] == rev[n - i:]:
        break
print()
for s in ["aacecaaa", "abcd", "", "a", "aab"]:
    a1, a2 = shortest_palindrome_brute(s), shortest_palindrome_rev_compare(s)
    print(f"{s!r:<11} -> {a1!r:<12} {a2!r}")
    assert a1 == a2

rev = aaacecaa
i=8: s[:i]=aacecaaa  rev[n-i:]=aaacecaa  bằng nhau? False
i=7: s[:i]=aacecaa   rev[n-i:]=aacecaa   bằng nhau? True

'aacecaaa'  -> 'aaacecaaa'  'aaacecaaa'
'abcd'      -> 'dcbabcd'    'dcbabcd'
''          -> ''           ''
'a'         -> 'a'          'a'
'aab'       -> 'baab'       'baab'


### Brute force chậm tới mức nào? Đếm thử!

Mỗi lần thử `i`, phép so sánh chuỗi dài `i` so **từng cặp ký tự** cho tới khi gặp cặp lệch.

- Trường hợp dễ: lệch ngay cặp đầu tiên → mỗi `i` tốn 1 phép so.
- Trường hợp khó: tiền tố "**gần như** palindrome" — so được rất xa rồi mới lệch, và chuyện đó lặp lại với **rất nhiều** `i`.

Input xấu: `s = "a"*m + "b" + "a"*(m-1)` (độ dài `n = 2m`). Với mỗi `i` từ `n` xuống `m + 1`, phép so phải chạy qua cả dãy chữ `a` rồi mới đụng chữ `b`. Tiền tố palindrome dài nhất là `"a"*m`, tức `L = m`.

Cell dưới viết tay vòng so sánh để **đếm** số lần so 2 ký tự. Quan sát: **`n` gấp đôi → số phép so gấp khoảng 4** — dấu hiệu kinh điển của `O(n²)`.

In [6]:
def count_char_comparisons(s: str):
    """Mô phỏng brute force 'so s[:i] với rev[n-i:]' và ĐẾM số lần so 2 ký tự."""
    n, rev = len(s), s[::-1]
    comparisons = 0
    for i in range(n, 0, -1):
        same = True
        for k in range(i):
            comparisons += 1
            if s[k] != rev[n - i + k]:
                same = False
                break
        if same:
            return i, comparisons
    return 0, comparisons


prev = None
for m in [100, 200, 400, 800]:
    s = "a" * m + "b" + "a" * (m - 1)
    L, cnt = count_char_comparisons(s)
    ratio = "" if prev is None else f"  (gấp {cnt / prev:.2f} lần dòng trên)"
    print(f"n={len(s):>5}  L={L:>4}  số phép so sánh={cnt:>7}{ratio}")
    prev = cnt

n=  200  L= 100  số phép so sánh=   5150
n=  400  L= 200  số phép so sánh=  20300  (gấp 3.94 lần dòng trên)
n=  800  L= 400  số phép so sánh=  80600  (gấp 3.97 lần dòng trên)
n= 1600  L= 800  số phép so sánh= 321200  (gấp 3.99 lần dòng trên)


### Độ phức tạp của Cách 1 và nút thắt

Gọi `n = len(s)`.

- **Time**: xấu nhất `O(n²)`. Tổng quát số phép so không vượt `n + (n-1) + ... + 1 = n(n+1)/2`; với input trên là khoảng `n²/8` (`n = 1600` → 321 200 phép so).
- **Space**: `O(n)` cho `rev` và các lát cắt tạm.

Ghi chú thực tế: trong Python, phép `==` giữa hai chuỗi chạy bằng C nên bản `s[:i] == rev[n-i:]` nhanh hơn rất nhiều so với vòng lặp Python thuần ở cell trên; nhiều người báo bản này vẫn có thể qua LeetCode. Nhưng **đừng dựa vào điều đó**: về lý thuyết nó vẫn `O(n²)` (với `n = 5 * 10^4`, input xấu tốn cỡ hàng trăm triệu phép so ký tự), và trong phỏng vấn người ta chờ đợi lời giải tốt hơn.

**Nút thắt**: khi phép so ở bước `i` thất bại, ta đã biết khá nhiều về chuỗi (ví dụ: "đầu chuỗi là một dãy dài chữ `a`"), nhưng sang bước `i - 1` lại **so lại từ đầu**, vứt hết thông tin cũ. Có hai hướng cải tiến:

1. **Hai con trỏ** (Cách 2): quét một lượt để **loại nhanh** phần đuôi chắc chắn không thuộc tiền tố palindrome.
2. **KMP** (Cách 3): **tái sử dụng** thông tin đã so khớp, không bao giờ so lại từ đầu → `O(n)` được bảo đảm.

## 7. Kiến thức nền 2 — Hai con trỏ (two pointers) kiểm tra palindrome

Thay vì tạo chuỗi đảo, ta dùng **hai chỉ số** (gọi là "con trỏ" — pointer):

- `lo` bắt đầu ở đầu chuỗi (`0`), `hi` bắt đầu ở cuối (`len(t) - 1`).
- So `t[lo]` với `t[hi]`: bằng nhau thì `lo += 1`, `hi -= 1` (cùng tiến vào giữa); khác nhau thì **không** phải palindrome.
- Dừng khi `lo >= hi` (hai con trỏ gặp nhau hoặc vượt qua nhau) → là palindrome.

Ví dụ `"abcba"`: cặp `(0, 4)`: `a = a`; cặp `(1, 3)`: `b = b`; `lo = hi = 2` → dừng → **True**.
Ví dụ `"abca"`: cặp `(0, 3)`: `a = a`; cặp `(1, 2)`: `b ≠ c` → **False**.

Ưu điểm: bộ nhớ `O(1)` — không tạo chuỗi mới. Ở Cách 2, hai con trỏ được dùng theo một kiểu **lạ hơn**: một con trỏ đi xuôi, một con trỏ đi ngược, và con trỏ xuôi **chỉ tiến khi khớp**.

In [7]:
def is_pal_two_pointers(t: str, verbose: bool = False) -> bool:
    lo, hi = 0, len(t) - 1
    while lo < hi:
        if verbose:
            verdict = "bằng nhau, đi tiếp" if t[lo] == t[hi] else "KHÁC -> không palindrome"
            print(f"  lo={lo} ({t[lo]})  hi={hi} ({t[hi]})  -> {verdict}")
        if t[lo] != t[hi]:
            return False
        lo += 1
        hi -= 1
    return True


for t in ["abcba", "abca"]:
    print(f"Kiểm tra {t!r}:")
    print("  =>", is_pal_two_pointers(t, verbose=True))

Kiểm tra 'abcba':
  lo=0 (a)  hi=4 (a)  -> bằng nhau, đi tiếp
  lo=1 (b)  hi=3 (b)  -> bằng nhau, đi tiếp
  => True
Kiểm tra 'abca':
  lo=0 (a)  hi=3 (a)  -> bằng nhau, đi tiếp
  lo=1 (b)  hi=2 (c)  -> KHÁC -> không palindrome
  => False


## 8. Kiến thức nền 3 — Đệ quy (recursion) và giới hạn độ sâu

**Đệ quy** là khi một hàm **gọi lại chính nó** trên một bài toán **nhỏ hơn**. Một hàm đệ quy luôn có 2 phần:

1. **Base case** (trường hợp dừng): bài đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Recursive case**: chia thành bài nhỏ hơn, gọi lại hàm, rồi dùng kết quả.

Ví dụ đảo chuỗi: `reverse("abc") = reverse("bc") + "a"`, `reverse("bc") = reverse("c") + "b"`, ..., và base case `reverse("") = ""`.

Mỗi lời gọi chưa xong phải **chờ** lời gọi con trả về; Python giữ các lời gọi đang chờ trong **call stack** (ngăn xếp lời gọi). Ngăn xếp này có **giới hạn độ sâu** (recursion limit — Python thuần thường là 1000, môi trường Jupyter có thể đặt cao hơn). Vượt giới hạn → lỗi **`RecursionError`**. Hãy nhớ điểm này: Cách 2 sẽ vấp đúng chỗ đó.

In [8]:
import sys


def reverse_recursive(t: str, depth: int = 0) -> str:
    print("  " * depth + f"gọi reverse_recursive({t!r})")
    if t == "":                                   # base case
        return ""
    result = reverse_recursive(t[1:], depth + 1) + t[0]   # bài nhỏ hơn: bỏ ký tự đầu
    print("  " * depth + f"-> trả về {result!r}")
    return result


print("Kết quả:", reverse_recursive("abc"))
print("Giới hạn độ sâu đệ quy hiện tại:", sys.getrecursionlimit())

gọi reverse_recursive('abc')
  gọi reverse_recursive('bc')
    gọi reverse_recursive('c')
      gọi reverse_recursive('')
    -> trả về 'c'
  -> trả về 'cb'
-> trả về 'cba'
Kết quả: cba
Giới hạn độ sâu đệ quy hiện tại: 1000


## 9. Cách 2 — Hai con trỏ "quét ngược" + đệ quy (cách trong editorial)

Đoạn code lõi rất ngắn:

```python
i = 0
for j in range(n - 1, -1, -1):   # j đi từ cuối về đầu chuỗi
    if s[i] == s[j]:
        i += 1                    # i chỉ tiến khi khớp
```

**Nó đang làm gì?** `j` đi từ cuối về đầu của `s`, tức là đang **đọc `rev(s)` từ trái sang phải**. Con trỏ `i` chỉ vào ký tự `s[i]` "đang cần tìm": mỗi khi gặp đúng ký tự đó, ta "tích" nó và chuyển sang tìm `s[i+1]`.

Đây chính là thuật toán kiểm tra **dãy con** (subsequence): một chuỗi `A` là dãy con của `B` nếu xóa bớt vài ký tự của `B` (không đổi thứ tự) thì ra `A`. Ví dụ `"ace"` là dãy con của `"abcde"` (xóa `b`, `d`); `"aec"` thì **không** (sai thứ tự).

Kết thúc vòng lặp: **`i` = độ dài tiền tố dài nhất của `s` là dãy con của `rev(s)`**. Tìm kiểu "gặp là lấy ngay" (tham lam — greedy) không bao giờ thiệt: lấy càng sớm thì phần `rev(s)` còn lại để tìm tiếp càng dài.

Trace với `s = "aacecaaa"` (`rev = "aaacecaa"`):

| `j` | `s[j]` | đang cần `s[i]` | Kết quả |
|:-:|:-:|:-:|---|
| 7 | a | `s[0]` = a | khớp → `i = 1` |
| 6 | a | `s[1]` = a | khớp → `i = 2` |
| 5 | a | `s[2]` = c | lệch, giữ `i = 2` |
| 4 | c | `s[2]` = c | khớp → `i = 3` |
| 3 | e | `s[3]` = e | khớp → `i = 4` |
| 2 | c | `s[4]` = c | khớp → `i = 5` |
| 1 | a | `s[5]` = a | khớp → `i = 6` |
| 0 | a | `s[6]` = a | khớp → `i = 7` |

Kết quả `i = 7` — trùng với `L = 7`. Cell dưới in đúng bảng này.

In [9]:
def scan_prefix_in_reverse(s: str, verbose: bool = False) -> int:
    """Trả về i: s[:i] là dãy con (subsequence) dài nhất của reverse(s) theo kiểu tham lam."""
    i = 0
    for j in range(len(s) - 1, -1, -1):
        match = s[i] == s[j]
        if verbose:
            action = f"khớp -> i={i + 1}" if match else f"lệch, giữ i={i}"
            print(f"j={j} s[j]={s[j]} | cần s[{i}]={s[i]} | {action}")
        if match:
            i += 1
    return i


print("Kết quả i =", scan_prefix_in_reverse("aacecaaa", verbose=True))
print("Với 'abcd': i =", scan_prefix_in_reverse("abcd"))

j=7 s[j]=a | cần s[0]=a | khớp -> i=1
j=6 s[j]=a | cần s[1]=a | khớp -> i=2
j=5 s[j]=a | cần s[2]=c | lệch, giữ i=2
j=4 s[j]=c | cần s[2]=c | khớp -> i=3
j=3 s[j]=e | cần s[3]=e | khớp -> i=4
j=2 s[j]=c | cần s[4]=c | khớp -> i=5
j=1 s[j]=a | cần s[5]=a | khớp -> i=6
j=0 s[j]=a | cần s[6]=a | khớp -> i=7
Kết quả i = 7
Với 'abcd': i = 1


### Vì sao `i >= L`? (chìa khóa của Cách 2)

Gọi `P = s[:L]` là tiền tố palindrome dài nhất.

- Vì `P` là palindrome nên `reverse(P) = P`. Mà `rev(s)` **kết thúc bằng** `reverse(P)` (đảo cả chuỗi thì phần đầu chạy xuống cuối). Vậy **`P` nằm liền một khối ở cuối `rev(s)`** → chắc chắn là dãy con của `rev(s)`.
- Quét tham lam tìm được tiền tố-dãy-con **dài nhất**, nên tìm được ít nhất `L` ký tự: **`i >= L`**.

Hệ quả quan trọng: phần **`s[i:]` chắc chắn nằm ngoài** tiền tố palindrome dài nhất → phần này **chắc chắn phải được soi gương lên đầu**.

Khi nào `i = n`? Khi cả `s` là dãy con của `rev(s)`; hai chuỗi **cùng độ dài** nên điều đó chỉ xảy ra khi chúng bằng nhau → `s == rev(s)` → `s` là palindrome. (Và ngược lại.)

Nhưng chú ý: quét chỉ cho ta **cận trên** của `L` — `i` có thể **lớn hơn** `L`. Cell dưới có ví dụ `"aabababa"`: `i = 6` nhưng `L = 2`. Vì vậy ta chưa thể dừng sau một lần quét — cần thêm **đệ quy**.

In [10]:
for s in ["aacecaaa", "abcd", "aabba", "abcba", "aabababa"]:
    i = scan_prefix_in_reverse(s)
    L = longest_pal_prefix_brute(s)
    note = "(i > L!)" if i > L else ""
    print(f"s={s:<9} i={i}  L={L}  phần chắc chắn phải soi gương s[i:]={s[i:]!r:<6} {note}")

s=aacecaaa  i=7  L=7  phần chắc chắn phải soi gương s[i:]='a'    
s=abcd      i=1  L=1  phần chắc chắn phải soi gương s[i:]='bcd'  
s=aabba     i=2  L=2  phần chắc chắn phải soi gương s[i:]='bba'  
s=abcba     i=5  L=5  phần chắc chắn phải soi gương s[i:]=''     
s=aabababa  i=6  L=2  phần chắc chắn phải soi gương s[i:]='ba'   (i > L!)


### Công thức đệ quy

Vì `L <= i`, tiền tố palindrome dài nhất của `s[:i]` **vẫn chính là `s[:L]`**: nó nằm trọn trong `s[:i]`, và `s[:i]` không thể có tiền tố palindrome dài hơn `L` (vì đó cũng sẽ là tiền tố palindrome của `s`). Gọi `f(x)` là đáp án cho chuỗi `x`, ta có:

**`f(s) = reverse(s[i:]) + f(s[:i]) + s[i:]`**

Kiểm tra bằng "đại số chuỗi": `f(s[:i]) = reverse(s[L:i]) + s[:i]`. Ghép vào:

`reverse(s[i:]) + reverse(s[L:i]) + s[:i] + s[i:]` = `reverse(s[L:]) + s` ✔ (vì đảo ngược của `s[L:i] + s[i:]` là `reverse(s[i:]) + reverse(s[L:i])`).

- **Base case**: `i == n` → `s` đã là palindrome → trả về `s` (bao gồm cả `s = ""`, khi đó `i = 0 = n`).
- **Luôn tiến bộ**: `s` không rỗng thì `L >= 1` nên `i >= 1`; nếu `i < n` thì bài con `s[:i]` **ngắn hơn hẳn** → đệ quy chắc chắn dừng.

In [11]:
def shortest_palindrome_recursive(s: str) -> str:
    i = scan_prefix_in_reverse(s)
    if i == len(s):                 # base case: s đã là palindrome (kể cả s rỗng)
        return s
    remain = s[i:]                  # chắc chắn nằm ngoài tiền tố palindrome dài nhất
    return remain[::-1] + shortest_palindrome_recursive(s[:i]) + remain


for s in ["aacecaaa", "abcd", "", "aabababa", "aabba"]:
    ans = shortest_palindrome_recursive(s)
    print(f"{s!r:<11} -> {ans!r}")
    assert ans == shortest_palindrome_brute(s)

'aacecaaa'  -> 'aaacecaaa'
'abcd'      -> 'dcbabcd'
''          -> ''
'aabababa'  -> 'abababaabababa'
'aabba'     -> 'abbaabba'


### Trace đệ quy với `s = "aabababa"` (ca có `i > L`)

| Tầng | Chuỗi đang xét | `i` sau khi quét | `remain = s[i:]` | Việc làm |
|:-:|---|:-:|:-:|---|
| 0 | `aabababa` | 6 | `"ba"` | đệ quy với `"aababa"` |
| 1 | `aababa` | 4 | `"ba"` | đệ quy với `"aaba"` |
| 2 | `aaba` | 2 | `"ba"` | đệ quy với `"aa"` |
| 3 | `aa` | 2 = n | — | đã palindrome → trả về `"aa"` |

Khi "quay về", mỗi tầng kẹp kết quả của tầng con giữa `reverse(remain) = "ab"` và `remain = "ba"`:

- Tầng 2: `"ab" + "aa" + "ba" = "abaaba"`
- Tầng 1: `"ab" + "abaaba" + "ba" = "ababaababa"`
- Tầng 0: `"ab" + "ababaababa" + "ba" = "abababaabababa"`

Kết quả trùng với công thức `reverse(s[2:]) + s = "ababab" + "aabababa"`.

In [12]:
def trace_recursive(s: str, depth: int = 0) -> str:
    pad = "    " * depth
    i = scan_prefix_in_reverse(s)
    if i == len(s):
        print(f"{pad}tầng {depth}: s={s!r}, i={i} = n -> palindrome, trả về {s!r}")
        return s
    remain = s[i:]
    print(f"{pad}tầng {depth}: s={s!r}, i={i}, remain={remain!r} -> đệ quy với {s[:i]!r}")
    middle = trace_recursive(s[:i], depth + 1)
    result = remain[::-1] + middle + remain
    print(f"{pad}tầng {depth}: {remain[::-1]!r} + {middle!r} + {remain!r} = {result!r}")
    return result


result = trace_recursive("aabababa")
assert result == shortest_palindrome_brute("aabababa")

tầng 0: s='aabababa', i=6, remain='ba' -> đệ quy với 'aababa'
    tầng 1: s='aababa', i=4, remain='ba' -> đệ quy với 'aaba'
        tầng 2: s='aaba', i=2, remain='ba' -> đệ quy với 'aa'
            tầng 3: s='aa', i=2 = n -> palindrome, trả về 'aa'
        tầng 2: 'ab' + 'aa' + 'ba' = 'abaaba'
    tầng 1: 'ab' + 'abaaba' + 'ba' = 'ababaababa'
tầng 0: 'ab' + 'ababaababa' + 'ba' = 'abababaabababa'


### Trường hợp xấu `"aa" + "ba"*k`: chậm `O(n²)` và `RecursionError`

Trace trên cho thấy mỗi tầng chỉ "gọt" được **2 ký tự**. Với `s = "aa" + "ba"*k` (độ dài `n = 2k + 2`):

- **Độ sâu đệ quy** là `k ≈ n/2`.
- Mỗi tầng quét **toàn bộ** chuỗi hiện tại: tổng công việc `n + (n-2) + (n-4) + ... ≈ n²/4` → **`O(n²)`**.
- Với `n = 5 * 10^4`, độ sâu cỡ 25 000 — vượt xa giới hạn đệ quy của Python → `RecursionError`.

**Cách sửa — khử đệ quy**: mỗi tầng chỉ làm đúng một việc: thêm `reverse(remain)` vào **bên trái** và `remain` vào **bên phải** kết quả của tầng con. Vậy ta dùng vòng `while`, gom các mảnh vào 2 list `left_parts`, `right_parts`, cuối cùng ghép lại. Chú ý thứ tự: mảnh của tầng ngoài cùng nằm **ngoài cùng** cả hai phía, nên bên phải phải ghép **theo thứ tự ngược** (`reversed(right_parts)` duyệt list từ cuối về đầu).

**Độ phức tạp Cách 2** (bản vòng lặp): Time thường rất nhanh (`"abcd"` chỉ cần 2 lần quét) nhưng **xấu nhất `O(n²)`**; Space `O(n)`. Cách 2 đẹp và ngắn, nhưng **không có bảo đảm `O(n)`**.

Cell dưới: đo độ sâu, rồi cố tình gây `RecursionError` (để chạy nhanh, ta **tạm hạ** giới hạn đệ quy xuống 300 rồi khôi phục lại ngay). Cú pháp `try: ... except RecursionError as e: ...` **bắt lỗi** để notebook không dừng lại; khối `finally:` luôn được chạy dù có lỗi hay không — chỗ lý tưởng để khôi phục giới hạn.

In [13]:
def recursion_depth(s: str) -> int:
    """Đếm số tầng đệ quy mà shortest_palindrome_recursive cần (đếm bằng vòng lặp)."""
    depth = 0
    while True:
        i = scan_prefix_in_reverse(s)
        if i == len(s):
            return depth
        s, depth = s[:i], depth + 1


for k in [5, 50, 500]:
    s = "aa" + "ba" * k
    print(f"k={k:>3}  n={len(s):>4}  độ sâu đệ quy = {recursion_depth(s)}")

old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(300)                     # tạm hạ giới hạn để demo nhanh
try:
    shortest_palindrome_recursive("aa" + "ba" * 1000)
except RecursionError as e:
    print("n = 2002, giới hạn 300 ->", type(e).__name__, ":", e)
finally:
    sys.setrecursionlimit(old_limit)           # luôn khôi phục
print("Đã khôi phục giới hạn:", sys.getrecursionlimit())

k=  5  n=  12  độ sâu đệ quy = 5
k= 50  n= 102  độ sâu đệ quy = 50
k=500  n=1002  độ sâu đệ quy = 500
n = 2002, giới hạn 300 -> RecursionError : maximum recursion depth exceeded
Đã khôi phục giới hạn: 1000


In [14]:
def shortest_palindrome_two_pointers(s: str) -> str:
    """Bản vòng lặp của Cách 2: cùng logic, không còn RecursionError."""
    left_parts, right_parts = [], []
    while True:
        i = scan_prefix_in_reverse(s)
        if i == len(s):                     # phần lõi đã là palindrome
            break
        remain = s[i:]
        left_parts.append(remain[::-1])     # tầng ngoài đứng ngoài cùng bên trái
        right_parts.append(remain)          # ... và ngoài cùng bên phải
        s = s[:i]
    return "".join(left_parts) + s + "".join(reversed(right_parts))


for s in ["aacecaaa", "abcd", "", "aabababa"]:
    ans = shortest_palindrome_two_pointers(s)
    print(f"{s!r:<11} -> {ans!r}")
    assert ans == shortest_palindrome_recursive(s)

big = "aa" + "ba" * 1000
assert shortest_palindrome_two_pointers(big) == shortest_palindrome_brute(big)
print("n =", len(big), "-> bản vòng lặp chạy được (độ sâu 1000 không còn là vấn đề), khớp brute force")

'aacecaaa'  -> 'aaacecaaa'
'abcd'      -> 'dcbabcd'
''          -> ''
'aabababa'  -> 'abababaabababa'
n = 2002 -> bản vòng lặp chạy được (độ sâu 1000 không còn là vấn đề), khớp brute force


## 10. Kiến thức nền 4 — Tiền tố, hậu tố và "biên" (border)

Nhớ lại mẹo ở Cách 1: **`s[:k]` là palindrome ⟺ `s[:k] == rev[n-k:]`** — tiền tố dài `k` của `s` bằng hậu tố dài `k` của `rev`. Nếu nối hai chuỗi lại thành một (mục 13), câu hỏi "tìm `L`" biến thành: *"chuỗi dài nhất vừa là **tiền tố** vừa là **hậu tố** là bao nhiêu?"* — đúng câu hỏi mà thuật toán **KMP** (Knuth–Morris–Pratt) trả lời trong `O(n)`.

Định nghĩa với một chuỗi `t`:

- **Tiền tố thực sự** (proper prefix): tiền tố **ngắn hơn** `t` (không tính chính `t`).
- **Hậu tố thực sự** (proper suffix): tương tự, hậu tố ngắn hơn `t`.
- **Biên** (border): chuỗi **không rỗng** vừa là tiền tố thực sự vừa là hậu tố thực sự của `t`.

| `t` | Các biên | Biên dài nhất |
|---|---|:-:|
| `"abab"` | `"ab"` | 2 |
| `"aabaa"` | `"a"`, `"aa"` | 2 |
| `"aaaa"` | `"a"`, `"aa"`, `"aaa"` | 3 |
| `"abc"` | (không có) | 0 |

Chú ý: biên ở đầu và biên ở cuối **được phép chồng lên nhau** — trong `"aaaa"`, bản `"aaa"` ở đầu (vị trí 0–2) và bản ở cuối (vị trí 1–3) dùng chung 2 ký tự.

In [15]:
def borders_naive(t: str) -> list:
    """Liệt kê mọi biên của t: thử từng độ dài k = 1..len(t)-1, so tiền tố với hậu tố."""
    m = len(t)
    return [t[:k] for k in range(1, m) if t[:k] == t[m - k:]]


for t in ["abab", "aabaa", "aaaa", "abc"]:
    print(f"{t!r:<8} biên = {borders_naive(t)}")

s = "aacecaaa"; rev = s[::-1]; n = len(s)
print("\nLiên hệ với bài: tiền tố s[:k] == hậu tố rev[n-k:] đúng với k =",
      [k for k in range(1, n + 1) if s[:k] == rev[n - k:]])

'abab'   biên = ['ab']
'aabaa'  biên = ['a', 'aa']
'aaaa'   biên = ['a', 'aa', 'aaa']
'abc'    biên = []

Liên hệ với bài: tiền tố s[:k] == hậu tố rev[n-k:] đúng với k = [1, 2, 7]


## 11. Mảng `lps` (prefix function) — "trái tim" của KMP

Với chuỗi `t` độ dài `m`, định nghĩa:

> **`lps[i]`** = độ dài **biên dài nhất** của tiền tố `t[0..i]` (tức `t[:i+1]`).

Tên `lps` viết tắt của *longest proper prefix which is also suffix*; một số tài liệu gọi là **prefix function**, ký hiệu `π`.

Tính tay với `t = "aabaaab"`:

| `i` | `t[0..i]` | Biên dài nhất | `lps[i]` |
|:-:|---|---|:-:|
| 0 | `a` | (không có — 1 ký tự không có tiền tố thực sự khác rỗng) | 0 |
| 1 | `aa` | `a` | 1 |
| 2 | `aab` | (không có) | 0 |
| 3 | `aaba` | `a` | 1 |
| 4 | `aabaa` | `aa` | 2 |
| 5 | `aabaaa` | `aa` | 2 |
| 6 | `aabaaab` | `aab` | 3 |

→ `lps = [0, 1, 0, 1, 2, 2, 3]`.

Trước khi học cách tính nhanh, ta viết bản **theo đúng định nghĩa** (chậm nhưng chắc chắn đúng) để làm chuẩn đối chiếu.

In [16]:
def lps_naive(t: str) -> list:
    """Theo đúng định nghĩa: với mỗi i, thử mọi độ dài biên k. Chậm (cỡ O(m^3)) nhưng chắc chắn đúng."""
    lps = []
    for i in range(len(t)):
        prefix = t[:i + 1]
        best = 0
        for k in range(1, i + 1):                    # k < len(prefix) = i + 1
            if prefix[:k] == prefix[i + 1 - k:]:
                best = k
        lps.append(best)
    return lps


print(lps_naive("aabaaab"))

[0, 1, 0, 1, 2, 2, 3]


## 12. Tính `lps` nhanh: hai ý tưởng

Giả sử đã biết `lps[0..i-1]`. Gọi **`j = lps[i-1]`** — độ dài biên dài nhất của `t[0..i-1]`. Ta muốn tính `lps[i]`.

### Ý tưởng 1 — Biên chỉ dài thêm được tối đa 1

Lấy một biên bất kỳ của `t[0..i]` và bỏ ký tự cuối của nó: ta được một biên (hoặc chuỗi rỗng) của `t[0..i-1]`. Vậy **`lps[i] <= lps[i-1] + 1`**.

Ứng viên tốt nhất là **kéo dài biên cũ thêm 1 ký tự**. Biên cũ là `t[0..j-1]`, ký tự đứng **ngay sau** nó ở đầu chuỗi là `t[j]`. Nếu **`t[i] == t[j]`** thì `lps[i] = j + 1`.

Ví dụ `t = "aabaaab"`, `i = 6`: `j = lps[5] = 2` (biên `"aa"`); `t[6] = 'b'` và `t[2] = 'b'` → khớp → `lps[6] = 3` (biên `"aab"`).

### Ý tưởng 2 — Lệch thì nhảy sang "biên của biên"

Nếu `t[i] != t[j]`, ta cần **biên ngắn hơn kế tiếp** của `t[0..i-1]` để thử kéo dài. Biên đó ở đâu?

- Biên `t[0..j-1]` xuất hiện **cả ở đầu lẫn ở cuối** `t[0..i-1]`.
- Một biên ngắn hơn của `t[0..i-1]` thì vừa là tiền tố, vừa là hậu tố của **chính đoạn `t[0..j-1]`** — tức là một biên của biên!
- Biên dài nhất của `t[0..j-1]` có độ dài `lps[j-1]` — đã tính rồi. Vậy: **`j = lps[j-1]`** rồi thử lại.

Lặp lại việc nhảy (`while`) cho tới khi khớp, hoặc `j = 0` (không còn biên nào để thử).

Ví dụ `t = "aabaaab"`, `i = 5`: `j = lps[4] = 2` (biên `"aa"` của `"aabaa"`); `t[5] = 'a'` vs `t[2] = 'b'` → lệch → nhảy `j = lps[1] = 1` (biên `"a"` của `"aa"`); `t[5] = 'a'` vs `t[1] = 'a'` → khớp → `lps[5] = 2`.

Cell dưới kiểm chứng Ý tưởng 2: đi theo chuỗi nhảy `lps[m-1] → lps[k-1] → ...` liệt kê **đúng và đủ** mọi biên, từ dài đến ngắn — nên khi "nhảy", ta không bỏ sót biên nào.

In [17]:
def border_chain(t: str) -> list:
    """Đi theo chuỗi nhảy k = lps[m-1], lps[k-1], ... để liệt kê độ dài MỌI biên của t."""
    lps = lps_naive(t)
    k, lengths = lps[-1], []
    while k > 0:
        lengths.append(k)
        k = lps[k - 1]          # biên dài nhất của chính biên hiện tại
    return lengths


for t in ["aabaa", "aaaa", "abacaba", "aabaaab"]:
    direct = [len(b) for b in borders_naive(t)][::-1]
    print(f"{t!r:<10} chuỗi nhảy = {border_chain(t)!s:<10} liệt kê trực tiếp = {direct}")
    assert border_chain(t) == direct

'aabaa'    chuỗi nhảy = [2, 1]     liệt kê trực tiếp = [2, 1]
'aaaa'     chuỗi nhảy = [3, 2, 1]  liệt kê trực tiếp = [3, 2, 1]
'abacaba'  chuỗi nhảy = [3, 1]     liệt kê trực tiếp = [3, 1]
'aabaaab'  chuỗi nhảy = [3]        liệt kê trực tiếp = [3]


### Ghép hai ý tưởng thành hàm `compute_lps`

```python
lps = [0] * m          # lps[0] = 0 luôn đúng (1 ký tự không có biên)
j = 0                  # j = độ dài biên dài nhất của t[0..i-1]
for i in range(1, m):
    while j > 0 and t[i] != t[j]:
        j = lps[j - 1]         # Ý tưởng 2: lùi về biên của biên
    if t[i] == t[j]:
        j += 1                 # Ý tưởng 1: kéo dài thêm 1
    lps[i] = j
```

Giải thích từng chi tiết:

- `i` bắt đầu từ `1` vì `lps[0] = 0` đã biết.
- `j` được "mang" từ vòng trước sang vòng sau: đầu vòng `i`, `j` chính là `lps[i-1]` — không cần đọc lại mảng.
- Điều kiện `j > 0` trong `while`: khi `j = 0` thì không còn biên nào để lùi (và `lps[j-1]` sẽ thành `lps[-1]` — sai hoàn toàn).
- Sau `while`: hoặc `t[i] == t[j]` (khớp → `j += 1`), hoặc `j = 0` và `t[i] != t[0]` (không có biên → `lps[i] = 0`).

In [18]:
import random


def compute_lps(t: str) -> list:
    m = len(t)
    lps = [0] * m
    j = 0
    for i in range(1, m):
        while j > 0 and t[i] != t[j]:
            j = lps[j - 1]
        if t[i] == t[j]:
            j += 1
        lps[i] = j
    return lps


print(compute_lps("aabaaab"))
random.seed(214)                 # cố định hạt giống: chạy lại vẫn ra đúng các chuỗi này
for _ in range(500):
    # random.randint(1, 12): số nguyên ngẫu nhiên từ 1 đến 12 (độ dài chuỗi)
    # random.choice("ab"): chọn ngẫu nhiên 1 ký tự trong "ab"
    t = "".join(random.choice("ab") for _ in range(random.randint(1, 12)))
    assert compute_lps(t) == lps_naive(t), t
print("compute_lps khớp lps_naive trên 500 chuỗi ngẫu nhiên.")

[0, 1, 0, 1, 2, 2, 3]
compute_lps khớp lps_naive trên 500 chuỗi ngẫu nhiên.


### Trace `compute_lps("aabaaab")`

| `i` | `t[i]` | `j` đầu vòng | Nhảy (`while`) | So `t[i]` với `t[j]` | `lps[i]` |
|:-:|:-:|:-:|---|---|:-:|
| 1 | a | 0 | — | `t[0]` = a → khớp | 1 |
| 2 | b | 1 | lệch `t[1]` = a → `j = lps[0] = 0` | `t[0]` = a → vẫn lệch | 0 |
| 3 | a | 0 | — | `t[0]` = a → khớp | 1 |
| 4 | a | 1 | — | `t[1]` = a → khớp | 2 |
| 5 | a | 2 | lệch `t[2]` = b → `j = lps[1] = 1` | `t[1]` = a → khớp | 2 |
| 6 | b | 2 | — | `t[2]` = b → khớp | 3 |

Dòng `i = 5` là chỗ đáng xem nhất: biên `"aa"` không kéo dài được (`'a' ≠ 'b'`), ta lùi về biên `"a"` và kéo dài thành công.

In [19]:
def trace_lps(t: str) -> list:
    m = len(t)
    lps = [0] * m
    j = 0
    print(f"t = {t!r}, lps[0] = 0")
    for i in range(1, m):
        line = f"i={i} t[i]={t[i]} j={j}"
        while j > 0 and t[i] != t[j]:
            line += f" | lệch t[{j}]={t[j]} -> j=lps[{j - 1}]={lps[j - 1]}"
            j = lps[j - 1]
        if t[i] == t[j]:
            line += f" | khớp t[{j}]={t[j]} -> j={j + 1}"
            j += 1
        else:
            line += f" | t[{j}]={t[j]} vẫn lệch, j=0"
        lps[i] = j
        print(line + f"  => lps[{i}]={j}")
    print("lps =", lps)
    return lps


lps_traced = trace_lps("aabaaab")
assert lps_traced == lps_naive("aabaaab")

t = 'aabaaab', lps[0] = 0
i=1 t[i]=a j=0 | khớp t[0]=a -> j=1  => lps[1]=1
i=2 t[i]=b j=1 | lệch t[1]=a -> j=lps[0]=0 | t[0]=a vẫn lệch, j=0  => lps[2]=0
i=3 t[i]=a j=0 | khớp t[0]=a -> j=1  => lps[3]=1
i=4 t[i]=a j=1 | khớp t[1]=a -> j=2  => lps[4]=2
i=5 t[i]=a j=2 | lệch t[2]=b -> j=lps[1]=1 | khớp t[1]=a -> j=2  => lps[5]=2
i=6 t[i]=b j=2 | khớp t[2]=b -> j=3  => lps[6]=3
lps = [0, 1, 0, 1, 2, 2, 3]


### Vì sao `compute_lps` chạy `O(m)` dù có `while` nằm trong `for`?

Hãy nhìn `j` như **mực nước trong bể**:

- Mỗi vòng `for`, `j` tăng **tối đa 1** (chỉ lệnh `j += 1`) → trong cả thuật toán, tổng mức tăng `<= m - 1`.
- Mỗi lần thân `while` chạy, `j` **giảm ít nhất 1** (vì `lps[j-1] <= j - 1 < j`).
- `j` không bao giờ âm.

→ Tổng số lần `while` chạy **không thể vượt** tổng mức tăng, tức `<= m - 1`. Tổng công việc `<= (m - 1) + (m - 1) = O(m)`.

Kiểu lập luận này gọi là **phân tích khấu hao** (amortized analysis): một vòng riêng lẻ có thể tốn nhiều — ví dụ `t = "a"*20 + "b"`, chữ `'b'` cuối cùng gây **19 lần nhảy liền** — nhưng đó chỉ là "tiêu" số lần tăng đã "tích" từ trước. Cell dưới đếm để kiểm chứng.

In [20]:
def lps_with_counters(t: str):
    m = len(t)
    lps = [0] * m
    j = inc = dec = 0
    for i in range(1, m):
        while j > 0 and t[i] != t[j]:
            j = lps[j - 1]
            dec += 1
        if t[i] == t[j]:
            j += 1
            inc += 1
        lps[i] = j
    return inc, dec


for t in ["a" * 20 + "b", "ab" * 10, "aabaaab" * 3, "abcdefghij" * 2]:
    inc, dec = lps_with_counters(t)
    m = len(t)
    print(f"m={m:>2}  số lần tăng={inc:>2}  số lần nhảy (while)={dec:>2}  tổng={inc + dec:>2} <= 2(m-1)={2 * (m - 1)}")
    assert dec <= inc <= m - 1

m=21  số lần tăng=19  số lần nhảy (while)=19  tổng=38 <= 2(m-1)=40
m=20  số lần tăng=18  số lần nhảy (while)= 0  tổng=18 <= 2(m-1)=38
m=21  số lần tăng=19  số lần nhảy (while)= 3  tổng=22 <= 2(m-1)=40
m=20  số lần tăng=10  số lần nhảy (while)= 0  tổng=10 <= 2(m-1)=38


## 13. Cách 3 — KMP: lời giải chính `O(n)`

Nối **`t = s + "#" + rev(s)`** (độ dài `m = 2n + 1`). Khẳng định: **`lps[m-1]` (phần tử cuối) chính là `L`.**

Lý do, từng bước:

1. `lps[m-1]` = độ dài biên dài nhất của **cả `t`**: chuỗi dài nhất vừa là tiền tố vừa là hậu tố thực sự của `t`.
2. **Biên không thể dài hơn `n`**: nếu độ dài `k > n` thì tiền tố dài `k` chứa `'#'` ở vị trí `n`, còn hậu tố dài `k` chứa `'#'` ở vị trí `k - n - 1` (đếm trong hậu tố). Hai chuỗi bằng nhau thì `'#'` phải ở cùng vị trí: `n = k - n - 1` ⟺ `k = 2n + 1 = m` — tức là cả `t`, không phải biên thực sự. Mâu thuẫn.
3. Với `k <= n`: tiền tố dài `k` của `t` là `s[:k]`; hậu tố dài `k` của `t` là `k` ký tự cuối của `rev(s)`, tức `reverse(s[:k])`. Hai cái bằng nhau ⟺ `s[:k] == reverse(s[:k])` ⟺ **`s[:k]` là palindrome**.

⟹ Biên dài nhất của `t` = tiền tố palindrome dài nhất của `s` = `L`. Sau đó: `answer = rev[:n-L] + s`.

Ví dụ `s = "abcd"`: `t = "abcd#dcba"`, biên dài nhất là `"a"` → `L = 1` → `"dcb" + "abcd"`.

### Vì sao bắt buộc phải có `"#"`?

Không có dấu ngăn cách, biên có thể **"vắt qua" chỗ nối** giữa `s` và `rev(s)` và dài hơn `n` — khi đó nó chẳng còn ý nghĩa "tiền tố palindrome" nữa:

- `s = "aaa"`: `t' = "aaaaaa"`, biên dài nhất `"aaaaa"` → 5 > n = 3.
- `s = "aaba"`: `t' = "aabaabaa"`, biên dài nhất `"aabaa"` (độ dài 5) — gồm cả `"aaba"` lẫn chữ `a` đầu tiên của `rev`. Trong khi `L` thật chỉ là 2 (`"aa"`).

`'#'` hoạt động như **bức tường**: nó không xuất hiện trong `s` (đề chỉ có chữ thường) nên không biên nào vượt qua được. Nếu `s` có thể chứa `'#'`, phải chọn ký tự ngăn cách khác — hoặc dùng cách so khớp không cần nối chuỗi (Bài tập 6).

In [21]:
for s in ["aaa", "aaba", "abcd"]:
    rev = s[::-1]
    n = len(s)
    L_no_sep = compute_lps(s + rev)[-1]
    L_sep = compute_lps(s + "#" + rev)[-1]
    print(f"s={s!r:<7} L thật={longest_pal_prefix_brute(s)}  lps cuối KHÔNG '#'={L_no_sep}  CÓ '#'={L_sep}")
    print(f"          không '#' -> {rev[:n - L_no_sep] + s!r:<10}  có '#' -> {rev[:n - L_sep] + s!r}")

s='aaa'   L thật=3  lps cuối KHÔNG '#'=5  CÓ '#'=3
          không '#' -> 'aaaa'      có '#' -> 'aaa'
s='aaba'  L thật=2  lps cuối KHÔNG '#'=5  CÓ '#'=2
          không '#' -> 'abaaaba'   có '#' -> 'abaaba'
s='abcd'  L thật=1  lps cuối KHÔNG '#'=1  CÓ '#'=1
          không '#' -> 'dcbabcd'   có '#' -> 'dcbabcd'


In [22]:
def shortest_palindrome_kmp(s: str) -> str:
    n = len(s)
    rev = s[::-1]
    t = s + "#" + rev
    L = compute_lps(t)[-1]          # tiền tố palindrome dài nhất của s
    return rev[:n - L] + s          # soi gương phần đuôi s[L:] lên trước


for s in ["aacecaaa", "abcd", "", "a", "aabababa"]:
    ans = shortest_palindrome_kmp(s)
    print(f"{s!r:<11} -> {ans!r}")
    assert ans == shortest_palindrome_brute(s)

'aacecaaa'  -> 'aaacecaaa'
'abcd'      -> 'dcbabcd'
''          -> ''
'a'         -> 'a'
'aabababa'  -> 'abababaabababa'


### Trace KMP trên ví dụ 1: `s = "aacecaaa"`

`t = "aacecaaa" + "#" + "aaacecaa"` = `"aacecaaa#aaacecaa"`, `m = 17`. Cell dưới in từng vị trí `i` với:

- `t[i]` và **vùng** của nó: thuộc `s`, là `#`, hay thuộc `rev`;
- `lps[i]` và **biên** tương ứng `t[:lps[i]]`.

Dự đoán trước khi chạy: tại `'#'` (`i = 8`) thì `lps` phải bằng 0 ("bức tường"); trong vùng `rev`, `lps` sẽ tăng dần và kết thúc ở `7`.

In [23]:
def trace_kmp_shortest(s: str) -> str:
    n, rev = len(s), s[::-1]
    t = s + "#" + rev
    lps = compute_lps(t)
    print(f"s = {s!r}, rev = {rev!r}, t = {t!r} (m = {len(t)})")
    for i, ch in enumerate(t):           # enumerate cho từng cặp (vị trí i, ký tự ch)
        zone = "s" if i < n else ("#" if i == n else "rev")
        border = t[:lps[i]] if lps[i] else "-"
        print(f"i={i:>2}  t[i]={ch}  vùng={zone:<3}  lps={lps[i]}  biên={border}")
    L = lps[-1]
    ans = rev[:n - L] + s
    print(f"L = lps[-1] = {L} -> thêm rev[:{n - L}] = {rev[:n - L]!r} -> {ans!r}")
    return ans


ans = trace_kmp_shortest("aacecaaa")
assert ans == "aaacecaaa"

s = 'aacecaaa', rev = 'aaacecaa', t = 'aacecaaa#aaacecaa' (m = 17)
i= 0  t[i]=a  vùng=s    lps=0  biên=-
i= 1  t[i]=a  vùng=s    lps=1  biên=a
i= 2  t[i]=c  vùng=s    lps=0  biên=-
i= 3  t[i]=e  vùng=s    lps=0  biên=-
i= 4  t[i]=c  vùng=s    lps=0  biên=-
i= 5  t[i]=a  vùng=s    lps=1  biên=a
i= 6  t[i]=a  vùng=s    lps=2  biên=aa
i= 7  t[i]=a  vùng=s    lps=2  biên=aa
i= 8  t[i]=#  vùng=#    lps=0  biên=-
i= 9  t[i]=a  vùng=rev  lps=1  biên=a
i=10  t[i]=a  vùng=rev  lps=2  biên=aa
i=11  t[i]=a  vùng=rev  lps=2  biên=aa
i=12  t[i]=c  vùng=rev  lps=3  biên=aac
i=13  t[i]=e  vùng=rev  lps=4  biên=aace
i=14  t[i]=c  vùng=rev  lps=5  biên=aacec
i=15  t[i]=a  vùng=rev  lps=6  biên=aaceca
i=16  t[i]=a  vùng=rev  lps=7  biên=aacecaa
L = lps[-1] = 7 -> thêm rev[:1] = 'a' -> 'aaacecaaa'


### Đọc kết quả trace

- `i = 0..7` (vùng `s`): `lps` chỉ là thông tin phụ về chính `s` (ví dụ `lps[6] = 2` vì `"aacecaa"` có biên `"aa"`).
- `i = 8` (`'#'`): `lps = 0` — đúng như dự đoán, không biên nào vượt qua bức tường.
- `i = 9..16` (vùng `rev`): `lps` đi `1, 2, 2, 3, 4, 5, 6, 7`. Tại `i = 11`, biên `"aa"` không kéo dài được (`t[11] = 'a'` nhưng `t[2] = 'c'`) nên nhảy về `"a"` rồi kéo dài lại thành `"aa"` — đó là lý do có hai số `2` liền nhau.
- `lps[16] = 7` → `L = 7`: tiền tố `"aacecaa"` là palindrome dài nhất → chỉ thêm `rev[:1] = "a"` → `"aaacecaaa"` ✔.

### Độ phức tạp của Cách 3

Gọi `n = len(s)`, `m = 2n + 1`.

- **Time**: `O(m) = O(n)` — tạo `rev` và `t` là `O(n)`, `compute_lps` là `O(m)` theo phân tích khấu hao ở mục 12, ghép đáp án `O(n)`.
- **Space**: `O(n)` cho `rev`, `t` và mảng `lps`.

Đây là lời giải có **bảo đảm** `O(n)` trong mọi trường hợp — không phụ thuộc may rủi như Cách 2 hay Cách 4.

## 14. Cách 4 (tùy chọn) — Rolling hash: so sánh "dấu vân tay"

Ý tưởng: thay vì so từng ký tự, ta biến mỗi tiền tố thành **một con số** ("dấu vân tay"), rồi so hai con số.

Hãy bắt đầu với chuỗi chữ số `"1213121"` — đọc tiền tố như một số nguyên:

- `fwd` = số đọc **xuôi** tiền tố; `bwd` = số đọc **ngược** tiền tố.
- Tiền tố là palindrome ⟺ `fwd == bwd` (đúng ý tưởng bài LeetCode 9 — Palindrome Number).

Cái hay: khi thêm chữ số mới `d` ở vị trí `i` (đếm từ 0), cả hai số cập nhật được trong **`O(1)`**:

- `fwd = fwd * 10 + d` — đẩy các chữ số cũ sang trái một hàng (giống hệt công thức đảo số ở bài 9);
- `bwd = bwd + d * 10^i` — chữ số mới trở thành chữ số **hàng cao nhất** của số đọc ngược.

| `i` | Tiền tố | `fwd` | `bwd` | Palindrome? |
|:-:|---|--:|--:|:-:|
| 0 | `1` | 1 | 1 | ✔ |
| 1 | `12` | 12 | 21 | ✘ |
| 2 | `121` | 121 | 121 | ✔ |
| 3 | `1213` | 1213 | 3121 | ✘ |
| 4 | `12131` | 12131 | 13121 | ✘ |
| 5 | `121312` | 121312 | 213121 | ✘ |
| 6 | `1213121` | 1213121 | 1213121 | ✔ |

In [24]:
digits = "1213121"
fwd = bwd = 0
power = 1                                   # 10^i
for i, ch in enumerate(digits):             # enumerate cho cặp (vị trí, ký tự)
    d = int(ch)
    fwd = fwd * 10 + d
    bwd = bwd + d * power
    power *= 10
    print(f"i={i}  tiền tố={digits[:i + 1]:<8} fwd={fwd:>8}  bwd={bwd:>8}  palindrome? {fwd == bwd}")

i=0  tiền tố=1        fwd=       1  bwd=       1  palindrome? True


i=1  tiền tố=12       fwd=      12  bwd=      21  palindrome? False
i=2  tiền tố=121      fwd=     121  bwd=     121  palindrome? True
i=3  tiền tố=1213     fwd=    1213  bwd=    3121  palindrome? False
i=4  tiền tố=12131    fwd=   12131  bwd=   13121  palindrome? False
i=5  tiền tố=121312   fwd=  121312  bwd=  213121  palindrome? False
i=6  tiền tố=1213121  fwd= 1213121  bwd= 1213121  palindrome? True


### Từ chữ số sang chữ cái: cơ số `B` và phép lấy dư `MOD`

1. **Ký tự → số**: `x = ord(ch) - ord('a') + 1` (`ord` cho mã số của ký tự; nhờ đó `'a' → 1`, `'b' → 2`, ..., `'z' → 26`).
2. **Cơ số `B` thay cho 10**: chọn `B = 131` (lớn hơn 26 nên mỗi chuỗi là một "số hệ cơ số `B`" riêng biệt). Ví dụ `"aba"`: `fwd = 1·B² + 2·B + 1`, `bwd = 1 + 2·B + 1·B²` → bằng nhau, đúng vì `"aba"` là palindrome.
3. **Lấy dư cho `MOD`**: với `n = 5 * 10^4`, số hệ cơ số 131 có tới 50 000 "chữ số" → phép tính rất chậm. Ta giữ mọi thứ theo dư của một số nguyên tố lớn `MOD = 10^9 + 7`, nên mỗi bước vẫn là `O(1)`.

**Cái giá: va chạm hash (collision).** Hai chuỗi khác nhau có thể cho cùng số dư. Ví dụ phóng đại với `MOD = 13`: chuỗi `"ab"` có `fwd = 1·131 + 2 = 133 ≡ 3` và `bwd = 1 + 2·131 = 263 ≡ 3 (mod 13)` → hash "tưởng" `"ab"` là palindrome. **Sai!**

Điểm mấu chốt: hash chỉ có thể sai **một chiều**:

- Hai chuỗi **bằng nhau** → hash **chắc chắn** bằng nhau (không bao giờ bỏ sót).
- Hash bằng nhau → hai chuỗi chỉ **rất có thể** bằng nhau (có thể là "dương tính giả").

Cách làm an toàn: ghi lại **mọi** độ dài `i` mà hash khớp (danh sách ứng viên), rồi **kiểm tra thật** từ ứng viên dài nhất trở xuống. `L` thật chắc chắn có trong danh sách → kết quả **luôn đúng**; với `MOD` lớn, ứng viên giả gần như không có → thời gian kỳ vọng `O(n)`.

**Độ phức tạp Cách 4**: Time `O(n)` kỳ vọng (về lý thuyết, nếu có rất nhiều ứng viên giả thì mỗi lần kiểm tra thật tốn `O(n)`, nhưng điều đó gần như không xảy ra với `MOD` lớn); Space `O(n)` cho danh sách ứng viên và lát cắt khi kiểm tra.

In [25]:
def palindrome_prefix_candidates(s: str, B: int = 131, MOD: int = 1_000_000_007) -> list:
    """Các độ dài i mà hash(s[:i]) đọc xuôi == hash đọc ngược (có thể lẫn ứng viên GIẢ)."""
    fwd = bwd = 0
    power = 1                          # B^i (mod MOD)
    candidates = []
    for i, ch in enumerate(s):
        x = ord(ch) - ord("a") + 1
        fwd = (fwd * B + x) % MOD
        bwd = (bwd + x * power) % MOD
        power = power * B % MOD
        if fwd == bwd:
            candidates.append(i + 1)
    return candidates


def shortest_palindrome_hash(s: str, B: int = 131, MOD: int = 1_000_000_007) -> str:
    for L in reversed(palindrome_prefix_candidates(s, B, MOD)):   # ứng viên dài nhất trước
        if s[:L] == s[:L][::-1]:                                  # kiểm tra thật -> loại ứng viên giả
            return s[L:][::-1] + s
    return s                                                      # chỉ xảy ra khi s rỗng


print("MOD lớn, s='aacecaaa' -> ứng viên:", palindrome_prefix_candidates("aacecaaa"))
print("MOD=13,  s='ab'       -> ứng viên:", palindrome_prefix_candidates("ab", MOD=13), " <- 2 là ứng viên GIẢ")
print("MOD=7,   s='abaab'    -> ứng viên:", palindrome_prefix_candidates("abaab", MOD=7), " (L thật = 3)")
for s in ["aacecaaa", "abcd", "", "aabababa"]:
    print(f"{s!r:<11} -> {shortest_palindrome_hash(s)!r}")
print("MOD=13 bị va chạm nhưng nhờ kiểm tra lại vẫn đúng: 'ab' ->", repr(shortest_palindrome_hash("ab", MOD=13)))
print("MOD=7  bị va chạm nhưng nhờ kiểm tra lại vẫn đúng: 'abaab' ->", repr(shortest_palindrome_hash("abaab", MOD=7)))

MOD lớn, s='aacecaaa' -> ứng viên: [1, 2, 7]
MOD=13,  s='ab'       -> ứng viên: [1, 2]  <- 2 là ứng viên GIẢ
MOD=7,   s='abaab'    -> ứng viên: [1, 3, 5]  (L thật = 3)
'aacecaaa'  -> 'aaacecaaa'
'abcd'      -> 'dcbabcd'
''          -> ''
'aabababa'  -> 'abababaabababa'
MOD=13 bị va chạm nhưng nhờ kiểm tra lại vẫn đúng: 'ab' -> 'bab'
MOD=7  bị va chạm nhưng nhờ kiểm tra lại vẫn đúng: 'abaab' -> 'baabaab'


## 15. Code nộp LeetCode (KMP)

Chuyển `shortest_palindrome_kmp` + `compute_lps` thành **một method** đúng signature `shortestPalindrome(self, s: str) -> str`:

1. `rev = s[::-1]`, `t = s + "#" + rev`.
2. Tính mảng `lps` của `t` (vòng `for` + `while` lùi về biên của biên).
3. `L = lps[-1]`, trả về `rev[:len(s) - L] + s`.

Chú ý:

- `s = ""` vẫn chạy đúng: `t = "#"`, `lps = [0]`, `L = 0`, trả về `"" + "" = ""`.
- Không có `print` trong code nộp; mọi thứ `O(n)` nên `n = 5 * 10^4` chạy rất nhanh.

In [26]:
class Solution:
    def shortestPalindrome(self, s: str) -> str:
        rev = s[::-1]
        t = s + "#" + rev                  # '#' không xuất hiện trong s (chỉ có chữ thường)
        m = len(t)
        lps = [0] * m
        j = 0
        for i in range(1, m):
            while j > 0 and t[i] != t[j]:
                j = lps[j - 1]             # lùi về biên của biên
            if t[i] == t[j]:
                j += 1                     # kéo dài biên thêm 1
            lps[i] = j
        L = lps[-1]                        # tiền tố palindrome dài nhất của s
        return rev[:len(s) - L] + s        # soi gương phần đuôi lên trước

In [27]:
import time

sol = Solution()
tests = [
    ("aacecaaa", "aaacecaaa"),          # ví dụ 1 của đề
    ("abcd", "dcbabcd"),                # ví dụ 2 của đề
    ("", ""),                           # chuỗi rỗng (constraints cho phép length = 0)
    ("a", "a"),                         # 1 ký tự
    ("aa", "aa"),                       # đã là palindrome
    ("ab", "bab"),
    ("aab", "baab"),
    ("aaba", "abaaba"),                 # bẫy quên '#'
    ("aabababa", "abababaabababa"),     # ca xấu của Cách 2
    ("abbacd", "dcabbacd"),
]
for s, expected in tests:
    got = sol.shortestPalindrome(s)
    assert got == expected, (s, got, expected)
    assert is_valid_answer(s, got)
    print(f"{s!r:<12} -> {got!r:<18} PASS")

# Input lớn nhất theo constraints (n = 5 * 10^4); time.perf_counter() đo thời gian thực
for name, s in [("toàn 'a'", "a" * 50000),
                ("'a'*m + 'b' + 'a'*(m-1)", "a" * 25000 + "b" + "a" * 24999),
                ("'aa' + 'ba'*k", "aa" + "ba" * 24999)]:
    start = time.perf_counter()
    got = sol.shortestPalindrome(s)
    elapsed = time.perf_counter() - start
    assert is_valid_answer(s, got)
    print(f"n={len(s)} ({name}): độ dài đáp án = {len(got)}, chạy dưới 1 giây? {elapsed < 1}")

'aacecaaa'   -> 'aaacecaaa'        PASS
'abcd'       -> 'dcbabcd'          PASS
''           -> ''                 PASS
'a'          -> 'a'                PASS
'aa'         -> 'aa'               PASS
'ab'         -> 'bab'              PASS
'aab'        -> 'baab'             PASS
'aaba'       -> 'abaaba'           PASS
'aabababa'   -> 'abababaabababa'   PASS
'abbacd'     -> 'dcabbacd'         PASS
n=50000 (toàn 'a'): độ dài đáp án = 50000, chạy dưới 1 giây? True


n=50000 ('a'*m + 'b' + 'a'*(m-1)): độ dài đáp án = 75000, chạy dưới 1 giây? True
n=50000 ('aa' + 'ba'*k): độ dài đáp án = 99998, chạy dưới 1 giây? True


## 16. Đối chiếu chéo tất cả các cách trên input ngẫu nhiên

Test tay chỉ bao được vài trường hợp; để chắc chắn hơn, ta sinh **3000 chuỗi ngẫu nhiên** và yêu cầu mọi cách cho **cùng một kết quả**:

- `random.seed(214)` cố định "hạt giống" → mỗi lần chạy lại notebook sinh ra **đúng** dữ liệu đó (dễ tái hiện lỗi).
- Dùng bảng chữ cái **nhỏ** (`"ab"` hoặc `"abc"`) để có nhiều palindrome dài — với 26 chữ cái thì chuỗi ngẫu nhiên hầu như không có palindrome dài, test sẽ "quá dễ".
- Với chuỗi ngắn (`n <= 7`), so thêm với **Cách 0** (thử mọi `X`) — oracle hoàn toàn độc lập, không dựa vào quan sát ở mục 5.
- Có cả bản hash với `MOD = 13` cố tình yếu: va chạm xảy ra thường xuyên nhưng nhờ bước kiểm tra lại, kết quả vẫn phải đúng.

In [28]:
random.seed(214)
methods = {
    "brute": shortest_palindrome_brute,
    "rev_compare": shortest_palindrome_rev_compare,
    "recursive": shortest_palindrome_recursive,
    "two_pointers": shortest_palindrome_two_pointers,
    "kmp": shortest_palindrome_kmp,
    "hash": shortest_palindrome_hash,
    "hash_MOD13": lambda s: shortest_palindrome_hash(s, MOD=13),   # lambda: hàm ngắn viết trên 1 dòng
    "Solution": Solution().shortestPalindrome,
}
oracle_checked = 0
for _ in range(3000):
    alphabet = random.choice(["ab", "abc"])
    s = "".join(random.choice(alphabet) for _ in range(random.randint(0, 14)))
    expected = shortest_palindrome_brute(s)
    assert is_valid_answer(s, expected)
    for name, f in methods.items():
        assert f(s) == expected, (name, s, f(s), expected)
    if len(s) <= 7:                                 # oracle thử-mọi-X: độc lập hoàn toàn
        assert shortest_palindrome_try_all_X(s)[0] == expected, s
        oracle_checked += 1
print(f"3000 chuỗi ngẫu nhiên: {len(methods)} cách cho cùng kết quả;",
      f"{oracle_checked} chuỗi ngắn khớp thêm oracle thử-mọi-X.")

3000 chuỗi ngẫu nhiên: 8 cách cho cùng kết quả; 1583 chuỗi ngắn khớp thêm oracle thử-mọi-X.


## 17. Vì sao đúng? Tổng hợp các invariant + bảng so sánh

### Invariant (bất biến) bằng lời

1. **Quy bài toán** (mục 5): thêm `k` ký tự được ⟺ `s[:n-k]` là palindrome, và khi đó `X` bị ép bằng `reverse(s[n-k:])`. Vì vậy đáp án luôn là `reverse(s[L:]) + s` với `L` = độ dài tiền tố palindrome dài nhất.
2. **Cách 1**: duyệt `i` giảm dần và dừng ở palindrome đầu tiên → tìm đúng `L` (không bỏ sót vì đã thử mọi độ dài lớn hơn).
3. **Cách 2**: sau mỗi lần quét, `i >= L` nên phần `s[i:]` gỡ ra chắc chắn thuộc phần phải soi gương; tiền tố palindrome dài nhất của chuỗi lõi còn lại **luôn vẫn là `s[:L]`**. Chuỗi lõi ngắn dần nên thuật toán dừng, và lúc dừng lõi chính là `s[:L]`.
4. **Cách 3 (KMP)**: đầu mỗi vòng `i`, `j = lps[i-1]`. Vòng `while` thử **mọi** biên của `t[0..i-1]` từ dài đến ngắn (chuỗi nhảy liệt kê đủ — mục 12), nên biên đầu tiên kéo dài được là biên dài nhất của `t[0..i]`. Dấu `'#'` chặn mọi biên dài hơn `n`, nên `lps[-1] = L`.
5. **Cách 4 (hash)**: không có "âm tính giả" → `L` thật luôn nằm trong danh sách ứng viên; bước kiểm tra thật loại bỏ "dương tính giả".

### Bảng so sánh

`n = len(s)`; `σ` = số chữ cái khác nhau trong `s`; `k` = số ký tự phải thêm.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 0. Thử mọi `X` | sinh mọi chuỗi `X` theo độ dài tăng dần | `O(σ^k · n)` — hàm mũ | `O(n)` | chỉ để hiểu đề / làm oracle |
| 1. Brute force | `i` từ `n` xuống 1, so `s[:i]` với `rev[n-i:]` | `O(n²)` xấu nhất | `O(n)` | dễ viết nhất |
| 2. Hai con trỏ + đệ quy | quét tham lam, gỡ đuôi `s[i:]`, lặp lại | thường nhanh, `O(n²)` xấu nhất | `O(n)` | bản đệ quy có thể `RecursionError` |
| **3. KMP** | `lps[-1]` của `s + "#" + rev` | **`O(n)`** | `O(n)` | **lời giải chính**, bảo đảm |
| 4. Rolling hash | so hash xuôi/ngược của từng tiền tố | `O(n)` kỳ vọng | `O(n)` | phải kiểm tra lại vì va chạm |
| (Manacher) | bán kính palindrome tại mọi tâm | `O(n)` | `O(n)` | mạnh nhưng dài, khó nhớ — chỉ nêu tên |

Ghi chú về **Manacher**: thuật toán này tìm palindrome dài nhất quanh **mọi tâm** trong `O(n)`; `L` chính là palindrome dài nhất "chạm" tới vị trí 0. Nó tổng quát hơn (giải được cả bài #5 Longest Palindromic Substring) nhưng code dài và dễ sai; với bài 214, KMP gọn hơn nhiều.

## 18. Lỗi thường gặp

### Lỗi 1: Ghép sai phía — thêm vào cuối

Viết `s + rev[:n-L]` thay vì `rev[:n-L] + s`. Đề yêu cầu thêm vào **trước**. Với `"abcd"` kết quả `"abcddcb"` thậm chí còn **không phải** palindrome.

### Lỗi 2: Cắt nhầm mảnh của `rev`

Mảnh cần thêm là **`n - L` ký tự đầu** của `rev`: `rev[:n-L]`. Viết `rev[L:]` là sai — với `"abcd"` ra `"cba" + "abcd"`.

Nguy hiểm ở chỗ: với ví dụ 1 `"aacecaaa"` (`L = 7`, `n = 8`), tình cờ `rev[7:] = rev[:1] = "a"` nên code sai **vẫn pass ví dụ 1**! Bài học: luôn test thêm ví dụ 2 và edge case. Mẹo nhớ: "phải thêm `n - L` ký tự" → mảnh thêm vào phải dài đúng `n - L`.

In [29]:
def wrong_append_back(s: str) -> str:
    n, rev = len(s), s[::-1]
    L = compute_lps(s + "#" + rev)[-1]
    return s + rev[:n - L]            # SAI: thêm vào cuối


def wrong_slice(s: str) -> str:
    n, rev = len(s), s[::-1]
    L = compute_lps(s + "#" + rev)[-1]
    return rev[L:] + s                # SAI: cắt nhầm mảnh


for s in ["aacecaaa", "abcd"]:
    for f in [wrong_append_back, wrong_slice, shortest_palindrome_kmp]:   # f.__name__ là tên hàm
        ans = f(s)
        ok = ans == shortest_palindrome_kmp(s)
        print(f"{f.__name__:<24} {s!r:<11} -> {ans!r:<12} đúng? {ok}")
    print()

wrong_append_back        'aacecaaa'  -> 'aacecaaaa'  đúng? False
wrong_slice              'aacecaaa'  -> 'aaacecaaa'  đúng? True
shortest_palindrome_kmp  'aacecaaa'  -> 'aaacecaaa'  đúng? True

wrong_append_back        'abcd'      -> 'abcddcb'    đúng? False
wrong_slice              'abcd'      -> 'cbaabcd'    đúng? False
shortest_palindrome_kmp  'abcd'      -> 'dcbabcd'    đúng? True



### Lỗi 3: Quên dấu `"#"`

Như mục 13: biên vắt qua chỗ nối nên `lps[-1]` có thể vượt `n`, đáp án hết còn ngắn nhất (`"aaba"` → `"abaaaba"` thay vì `"abaaba"`).

### Lỗi 4: Tính `lps` của riêng `s`

`compute_lps(s)[-1]` là biên dài nhất của `s` — chuỗi vừa ở **đầu** vừa ở **cuối** `s` — chẳng liên quan gì tới palindrome. Ví dụ `"abcd"` có biên 0 → thêm cả `"dcba"`, ra chuỗi dài 8 thay vì 7. Phải tính trên `s + "#" + rev`.

### Lỗi 5: Dùng `if` thay cho `while` khi lùi biên

Chỉ nhảy **một lần** rồi gán luôn `lps[i] = j` dù vẫn còn lệch. Ví dụ `t = "aaab"`: tại `i = 3`, nhảy từ `j = 2` về `j = 1` rồi `t[3] = 'b' ≠ t[1] = 'a'` vẫn lệch — bản `if` ghi luôn `lps[3] = 1` (sai, đúng là 0). Với bài 214, lỗi này có thể trả về chuỗi **không phải palindrome** (tìm bằng vét cạn: `s = "aaabaabaa"`).

### Các lỗi lặt vặt khác

- **Đệ quy sâu** với Cách 2 → `RecursionError` ở `n` lớn; dùng bản `while` (mục 9).
- **Quên chuỗi rỗng**: code đọc `s[0]` hoặc `lps[-1]` trên list rỗng sẽ lỗi. Bản KMP có `'#'` nên `t` không bao giờ rỗng.
- **Dùng `s[-k:]` với `k = 0`**: trả về cả chuỗi thay vì chuỗi rỗng (mục 3) — hãy viết `s[n-k:]` hoặc `rev[:k]`.
- **Ký tự ngăn cách có thể xuất hiện trong input**: ở bài này an toàn vì chỉ có chữ thường; bài khác phải chọn ký tự khác.
- **Nghĩ brute force là `O(n)` vì code chỉ 3 dòng**: `s[:i] == rev[n-i:]` bên trong vòng lặp tự nó đã là `O(n)` → tổng `O(n²)`.

In [30]:
def lps_wrong_if(t: str) -> list:
    m = len(t)
    lps = [0] * m
    j = 0
    for i in range(1, m):
        if j > 0 and t[i] != t[j]:        # SAI: chỉ nhảy MỘT lần
            j = lps[j - 1]
        if t[i] == t[j]:
            j += 1
        lps[i] = j
    return lps


# Lỗi 3: quên '#'
s = "aaba"; rev = s[::-1]
L_bad = compute_lps(s + rev)[-1]
print(f"Lỗi 3  s={s!r}: L sai={L_bad} -> {rev[:len(s) - L_bad] + s!r}, đúng: {shortest_palindrome_kmp(s)!r}")

# Lỗi 4: lps của riêng s
for s in ["abcd", "aacecaaa"]:
    rev = s[::-1]
    L_bad = compute_lps(s)[-1]
    print(f"Lỗi 4  s={s!r}: L sai={L_bad} -> {rev[:len(s) - L_bad] + s!r}, đúng: {shortest_palindrome_kmp(s)!r}")

# Lỗi 5: if thay cho while
print("Lỗi 5  lps('aaab'): đúng", compute_lps("aaab"), "| bản if:", lps_wrong_if("aaab"))
s = "aaabaabaa"; rev = s[::-1]
L_bad = lps_wrong_if(s + "#" + rev)[-1]
bad = rev[:len(s) - L_bad] + s
print(f"Lỗi 5  s={s!r}: L sai={L_bad} -> {bad!r} (palindrome? {is_palindrome(bad)})")
print(f"       đúng: {shortest_palindrome_kmp(s)!r}")

Lỗi 3  s='aaba': L sai=5 -> 'abaaaba', đúng: 'abaaba'
Lỗi 4  s='abcd': L sai=0 -> 'dcbaabcd', đúng: 'dcbabcd'
Lỗi 4  s='aacecaaa': L sai=2 -> 'aaacecaacecaaa', đúng: 'aaacecaaa'
Lỗi 5  lps('aaab'): đúng [0, 1, 2, 0] | bản if: [0, 1, 2, 1]
Lỗi 5  s='aaabaabaa': L sai=6 -> 'aabaaabaabaa' (palindrome? False)
       đúng: 'aabaabaaabaabaa'


## 19. Mẹo nhận dạng pattern

- Đề nói **"thêm ký tự vào đầu (hoặc cuối) để thành palindrome ngắn nhất"** → nghĩ ngay tới **tiền tố (hoặc hậu tố) palindrome dài nhất**.
- **"Tiền tố là palindrome"** ⟺ **"tiền tố của `s` bằng hậu tố của `reverse(s)`"** → nối `s + "#" + reverse(s)` và lấy `lps[-1]`.
- Câu hỏi dạng **"chuỗi dài nhất vừa là tiền tố vừa là hậu tố"** → mảng `lps` / prefix function của KMP (hoặc Z-function).
- **Mẹo ký tự ngăn cách**: muốn so tiền tố của `A` với các đoạn của `B` bằng `lps`/Z-function → xử lý chuỗi `A + "#" + B`, trong đó `'#'` không có trong cả hai.
- Cần **so nhiều đoạn con trong `O(1)`** → rolling hash; nhớ hash chỉ sai một chiều nên **kiểm tra lại** khi cần chắc chắn.
- Thấy **đệ quy mà mỗi tầng chỉ thêm mảnh vào hai bên** kết quả của tầng con → khử đệ quy bằng vòng lặp + 2 list.
- Thấy vòng `while` lồng trong `for` nhưng biến "chỉ tăng 1 mỗi vòng, mỗi lần while giảm ít nhất 1" → nghĩ tới **phân tích khấu hao**, tổng vẫn `O(n)`.

## 20. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| Biến thể: chỉ được thêm vào **cuối** | Đối xứng với bài này: tìm **hậu tố** palindrome dài nhất, nối `rev + "#" + s` (Bài tập 5). |
| #28 Find the Index of the First Occurrence in a String | Bài KMP gốc: tìm chuỗi mẫu trong văn bản bằng `lps`. |
| #1392 Longest Happy Prefix | Đúng bằng `s[:lps[-1]]` — "biên dài nhất" của chính `s`. |
| #459 Repeated Substring Pattern | `s` lặp từ một khối ⟺ `lps[-1] > 0` và `n % (n - lps[-1]) == 0`. |
| #5 Longest Palindromic Substring, #647 Palindromic Substrings | Palindrome theo tâm: expand around center hoặc Manacher. |
| #336 Palindrome Pairs (Hard) | Dùng lại ý "tiền tố/hậu tố là palindrome" + hash map các từ đảo ngược. |
| #1312 Minimum Insertion Steps to Make a String Palindrome (Hard) | Được chèn **bất kỳ đâu** → bài khác hẳn, giải bằng DP (dãy con palindrome dài nhất). |

Cell dưới chạy thử hai bài dùng `lps` trên chính `s` (#1392 và #459) để thấy công cụ vừa học dùng lại được ngay.

In [31]:
def longest_happy_prefix(s: str) -> str:            # LeetCode #1392
    return s[:compute_lps(s)[-1]] if s else ""


def repeated_substring_pattern(s: str) -> bool:     # LeetCode #459
    n = len(s)
    border = compute_lps(s)[-1]
    return border > 0 and n % (n - border) == 0


print("#1392:", repr(longest_happy_prefix("level")), repr(longest_happy_prefix("ababab")))
print("#459 :", repeated_substring_pattern("abab"), repeated_substring_pattern("aba"),
      repeated_substring_pattern("abcabcabcabc"))
assert longest_happy_prefix("level") == "l" and longest_happy_prefix("ababab") == "abab"
assert repeated_substring_pattern("abab") and not repeated_substring_pattern("aba")

#1392: 'l' 'abab'
#459 : True False True


## 21. Bài tập tự luyện

Làm theo thứ tự — bài sau dùng lại kỹ năng của bài trước. Gợi ý theo 3 mức ở mục 22, đáp án ở mục 23 (cuối notebook). **Các cell TODO chỉ định nghĩa hàm, không tự chạy test** — làm xong hãy bỏ dấu `#` trước các dòng `assert`.

### Bài 1 — Đoán output (không chạy code)

Cho `s = "abac"`.

- (a) Tiền tố palindrome dài nhất của `s` là gì, `L` bằng bao nhiêu?
- (b) Chuỗi `t = s + "#" + s[::-1]` là gì?
- (c) `Solution().shortestPalindrome("abac")` trả về gì?
- (d) Với `s = "aababa"`, `scan_prefix_in_reverse(s)` trả về `i` bằng bao nhiêu? `L` bằng bao nhiêu? Vì sao không mâu thuẫn?

### Bài 2 — Tính tay mảng `lps`

Với `s = "abab"`, ta có `t = "abab#baba"`. Lập bảng `i`, `t[i]`, `j` đầu vòng, các lần nhảy, `lps[i]` như mục 12. Từ `lps[-1]`, suy ra `L` và đáp án `shortestPalindrome("abab")`.

### Bài 3 — Điền TODO: hàm tính `lps`

Hoàn thiện thân vòng `for` theo 2 ý tưởng của mục 12.

In [32]:
def exercise_compute_lps(t: str) -> list:
    m = len(t)
    lps = [0] * m
    j = 0
    for i in range(1, m):
        # TODO 1: khi j > 0 và t[i] != t[j], lùi j về biên của biên
        # TODO 2: nếu t[i] == t[j], kéo dài biên thêm 1
        # TODO 3: ghi lps[i]
        pass
    return lps


# Tự làm xong rồi mới mở test:
# assert exercise_compute_lps("aabaaab") == [0, 1, 0, 1, 2, 2, 3]
# assert exercise_compute_lps("abab#baba") == [0, 0, 1, 2, 0, 0, 1, 2, 3]
# assert exercise_compute_lps("aaab") == [0, 1, 2, 0]

### Bài 4 — Tự viết lại lời giải KMP

Viết lại toàn bộ lời giải trong **một hàm**, không gọi `compute_lps` hay bất kỳ hàm nào đã viết ở trên. Cố gắng **không nhìn** mục 15.

In [33]:
def exercise_shortest_palindrome(s: str) -> str:
    # TODO 1: rev = đảo ngược s; t = s + "#" + rev
    # TODO 2: tự tính mảng lps của t
    # TODO 3: L = lps[-1]; trả về rev[:n - L] + s
    raise NotImplementedError


# assert exercise_shortest_palindrome("aacecaaa") == "aaacecaaa"
# assert exercise_shortest_palindrome("abcd") == "dcbabcd"
# assert exercise_shortest_palindrome("") == ""
# assert exercise_shortest_palindrome("aaba") == "abaaba"

### Bài 5 — Biến thể: chỉ được thêm ký tự vào CUỐI

Cho `s`, chỉ được **thêm ký tự vào cuối**. Trả về palindrome ngắn nhất có dạng `s + Y`.

| `s` | Kết quả |
|---|---|
| `"abcd"` | `"abcdcba"` |
| `"aacecaaa"` | `"aacecaaacecaa"` |
| `"aab"` | `"aabaa"` |
| `""` | `""` |

Câu hỏi phụ: làm sao giải bài này **chỉ bằng một dòng** dùng lại `Solution().shortestPalindrome`?

In [34]:
def exercise_shortest_palindrome_append(s: str) -> str:
    """Chỉ được thêm ký tự vào CUỐI s. Trả về palindrome ngắn nhất."""
    # TODO: tìm hậu tố palindrome dài nhất (độ dài L), rồi thêm đảo ngược của s[:n-L] vào cuối
    raise NotImplementedError


# assert exercise_shortest_palindrome_append("abcd") == "abcdcba"
# assert exercise_shortest_palindrome_append("aacecaaa") == "aacecaaacecaa"
# assert exercise_shortest_palindrome_append("aab") == "aabaa"
# assert exercise_shortest_palindrome_append("") == ""

### Bài 6 — Mini challenge: KMP so khớp, không cần nối chuỗi

Tìm `L` bằng **KMP so khớp** (KMP matching) thay vì nối `s + "#" + rev`:

- Coi `s` là **mẫu** (pattern), `rev = s[::-1]` là **văn bản** (text).
- Tính `lps` của **riêng `s`**, rồi đọc lần lượt từng ký tự của `rev`, duy trì `j` = độ dài tiền tố dài nhất của `s` khớp với **đuôi** phần văn bản đã đọc.
- Đọc hết văn bản, `j` chính là `L`. Hãy giải thích vì sao, và vì sao `j` không thể đạt `n` trước ký tự cuối cùng của văn bản.

Ưu điểm: không cần ký tự ngăn cách (dùng được cả khi `s` chứa `'#'`), mảng `lps` chỉ dài `n` thay vì `2n + 1`.

In [35]:
def exercise_longest_pal_prefix_matching(s: str) -> int:
    """Tìm L bằng KMP SO KHỚP: pattern = s, text = reverse(s). Không tạo chuỗi s + '#' + rev."""
    # TODO 1: lps = mảng lps của riêng s
    # TODO 2: j = 0; với mỗi ký tự c của s[::-1]:
    #           - while j > 0 và c != s[j]: j = lps[j - 1]
    #           - if c == s[j]: j += 1
    # TODO 3: trả về j
    raise NotImplementedError


# assert exercise_longest_pal_prefix_matching("aacecaaa") == 7
# assert exercise_longest_pal_prefix_matching("abcd") == 1
# assert exercise_longest_pal_prefix_matching("aaba") == 2
# assert exercise_longest_pal_prefix_matching("") == 0

## 22. Gợi ý theo mức

**Bài 1**
- Mức 1: liệt kê các tiền tố của `"abac"` từ dài đến ngắn và kiểm tra từng cái.
- Mức 2: đáp án = `reverse(s[L:]) + s`. Với (d), chạy tay bảng `j`/`i` như mục 9.
- Mức 3: (d) nhớ rằng quét chỉ cho **cận trên** `i >= L`.

**Bài 2**
- Mức 1: tại `'#'` thì `lps` chắc chắn bằng 0 — vì sao?
- Mức 2: trong vùng `rev = "baba"`, chữ `'b'` đầu tiên không khớp `t[0] = 'a'`, các chữ sau lần lượt kéo dài biên.
- Mức 3: `lps` có 9 phần tử, phần tử cuối bằng 3 → `L = 3`; thêm `rev[:1]`.

**Bài 3**
- Mức 1: hai ý tưởng — "lệch thì lùi về biên của biên", "khớp thì kéo dài thêm 1".
- Mức 2: lùi phải là `while`, không phải `if`; điều kiện có `j > 0`.
- Mức 3: `while j > 0 and t[i] != t[j]: j = lps[j - 1]`, sau đó `if t[i] == t[j]: j += 1`, cuối cùng `lps[i] = j`.

**Bài 4**
- Mức 1: ba bước: nối chuỗi → `lps` → ghép đáp án.
- Mức 2: mảnh thêm vào là `rev[:n - L]` (dài đúng `n - L`), đặt **trước** `s`.
- Mức 3: thử lại với `"aaba"` để chắc chắn không quên `'#'`.

**Bài 5**
- Mức 1: đối xứng với bài gốc: "tiền tố palindrome" ↔ "hậu tố palindrome".
- Mức 2: hậu tố `s[n-k:]` là palindrome ⟺ `rev[:k] == s[n-k:]` → nối `rev + "#" + s`, lấy `lps[-1]`.
- Mức 3: đáp án `s + rev[L:]`. Một dòng: đảo `s`, giải bài gốc, đảo kết quả: `Solution().shortestPalindrome(s[::-1])[::-1]`.

**Bài 6**
- Mức 1: đây là vòng KMP so khớp kinh điển (bài #28), chỉ khác là ta không dừng khi khớp hết mà lấy `j` cuối cùng.
- Mức 2: invariant: sau khi đọc `rev[0..p]`, `j` = độ dài tiền tố dài nhất của `s` bằng một hậu tố của `rev[0..p]`. Đọc hết thì hậu tố dài `j` của `rev` bằng `s[:j]` → palindrome (mục 13).
- Mức 3: `j = n` nghĩa là `s` xuất hiện trọn trong phần đã đọc; vì văn bản cũng chỉ dài `n`, điều đó chỉ có thể xảy ra ở ký tự cuối cùng.

## 23. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

**Bài 1**
- (a) Tiền tố `"abac"` ✘, `"aba"` ✔ → `L = 3`.
- (b) `t = "abac#caba"`.
- (c) `reverse(s[3:]) + s = "c" + "abac" = "cabac"`.
- (d) `i = 4` nhưng `L = 2` (`"aa"`). Không mâu thuẫn: Cách 2 chỉ bảo đảm `i >= L`; ta sẽ đệ quy tiếp với `s[:4] = "aaba"`.

**Bài 2** — `t = "abab#baba"`:

| `i` | `t[i]` | `j` đầu vòng | Nhảy | `lps[i]` |
|:-:|:-:|:-:|---|:-:|
| 0 | a | — | — | 0 |
| 1 | b | 0 | — (lệch `t[0]` = a) | 0 |
| 2 | a | 0 | — (khớp `t[0]`) | 1 |
| 3 | b | 1 | — (khớp `t[1]`) | 2 |
| 4 | # | 2 | `2 → lps[1] = 0`, vẫn lệch | 0 |
| 5 | b | 0 | — (lệch `t[0]` = a) | 0 |
| 6 | a | 0 | — (khớp `t[0]`) | 1 |
| 7 | b | 1 | — (khớp `t[1]`) | 2 |
| 8 | a | 2 | — (khớp `t[2]`) | 3 |

`lps = [0, 0, 1, 2, 0, 0, 1, 2, 3]` → `L = 3` (`"aba"`) → đáp án `rev[:1] + s = "b" + "abab" = "babab"`.

Cell dưới kiểm chứng Bài 1–2 và chạy đáp án Bài 3–4; cell sau đó là Bài 5–6.

In [36]:
# ---------- Kiểm chứng Bài 1 và Bài 2 ----------
assert longest_pal_prefix_brute("abac") == 3
assert "abac" + "#" + "abac"[::-1] == "abac#caba"
assert Solution().shortestPalindrome("abac") == "cabac"
assert scan_prefix_in_reverse("aababa") == 4 and longest_pal_prefix_brute("aababa") == 2
assert compute_lps("abab#baba") == [0, 0, 1, 2, 0, 0, 1, 2, 3]
assert Solution().shortestPalindrome("abab") == "babab"


# ---------- Đáp án Bài 3 ----------
def answer_compute_lps(t: str) -> list:
    m = len(t)
    lps = [0] * m
    j = 0
    for i in range(1, m):
        while j > 0 and t[i] != t[j]:
            j = lps[j - 1]
        if t[i] == t[j]:
            j += 1
        lps[i] = j
    return lps


# ---------- Đáp án Bài 4 ----------
def answer_shortest_palindrome(s: str) -> str:
    n = len(s)
    rev = s[::-1]
    t = s + "#" + rev
    lps = [0] * len(t)
    j = 0
    for i in range(1, len(t)):
        while j > 0 and t[i] != t[j]:
            j = lps[j - 1]
        if t[i] == t[j]:
            j += 1
        lps[i] = j
    return rev[:n - lps[-1]] + s


assert answer_compute_lps("aabaaab") == [0, 1, 0, 1, 2, 2, 3]
assert answer_compute_lps("abab#baba") == [0, 0, 1, 2, 0, 0, 1, 2, 3]
assert answer_compute_lps("aaab") == [0, 1, 2, 0]
assert answer_shortest_palindrome("aacecaaa") == "aaacecaaa"
assert answer_shortest_palindrome("abcd") == "dcbabcd"
assert answer_shortest_palindrome("") == ""
assert answer_shortest_palindrome("aaba") == "abaaba"
print("Bài 1-4: PASS")

Bài 1-4: PASS


In [37]:
# ---------- Đáp án Bài 5 ----------
def answer_shortest_palindrome_append(s: str) -> str:
    n = len(s)
    rev = s[::-1]
    L = answer_compute_lps(rev + "#" + s)[-1]   # hậu tố palindrome dài nhất của s
    return s + rev[L:]                           # rev[L:] = đảo ngược của s[:n-L]


def answer_shortest_palindrome_append_one_line(s: str) -> str:
    return Solution().shortestPalindrome(s[::-1])[::-1]


# ---------- Đáp án Bài 6 ----------
def answer_longest_pal_prefix_matching(s: str) -> int:
    lps = answer_compute_lps(s)       # lps của riêng pattern s
    j = 0
    for c in reversed(s):             # text = rev(s), đọc từ trái sang phải
        while j > 0 and c != s[j]:
            j = lps[j - 1]
        if c == s[j]:
            j += 1                    # j = n chỉ có thể xảy ra ở ký tự cuối cùng
    return j


assert answer_shortest_palindrome_append("abcd") == "abcdcba"
assert answer_shortest_palindrome_append("aacecaaa") == "aacecaaacecaa"
assert answer_shortest_palindrome_append("aab") == "aabaa"
assert answer_shortest_palindrome_append("") == ""
assert answer_longest_pal_prefix_matching("aacecaaa") == 7
assert answer_longest_pal_prefix_matching("abcd") == 1
assert answer_longest_pal_prefix_matching("aaba") == 2
assert answer_longest_pal_prefix_matching("") == 0

random.seed(2140)
for _ in range(1000):
    s = "".join(random.choice("abc") for _ in range(random.randint(0, 12)))
    y = answer_shortest_palindrome_append(s)
    assert is_palindrome(y) and y.startswith(s)
    assert y == answer_shortest_palindrome_append_one_line(s)
    assert answer_longest_pal_prefix_matching(s) == longest_pal_prefix_brute(s)
    assert answer_shortest_palindrome(s) == shortest_palindrome_brute(s)
print("Bài 5-6: PASS (kể cả 1000 chuỗi ngẫu nhiên)")

Bài 5-6: PASS (kể cả 1000 chuỗi ngẫu nhiên)


## 24. Template ngắn nên nhớ

**Công thức một dòng**: *đáp án = đảo ngược(phần đuôi nằm ngoài tiền tố palindrome dài nhất) + `s`*; và *`L` = `lps[-1]` của `s + "#" + reverse(s)`*.

```python
class Solution:
    def shortestPalindrome(self, s: str) -> str:
        rev = s[::-1]
        t = s + "#" + rev                  # '#': bức tường, không có trong s
        lps = [0] * len(t)
        j = 0
        for i in range(1, len(t)):
            while j > 0 and t[i] != t[j]:
                j = lps[j - 1]             # lệch -> lùi về biên của biên
            if t[i] == t[j]:
                j += 1                     # khớp -> kéo dài biên
            lps[i] = j
        return rev[:len(s) - lps[-1]] + s  # thêm đúng n - L ký tự lên trước
```

Khung `lps` dùng lại cho mọi bài "vừa là tiền tố vừa là hậu tố" (#28, #459, #1392, ...):

```python
lps, j = [0] * m, 0
for i in range(1, m):
    while j and t[i] != t[j]: j = lps[j - 1]
    if t[i] == t[j]: j += 1
    lps[i] = j
```

Bản brute force dễ nhớ (để kiểm tra chéo, `O(n²)`):

```python
rev = s[::-1]
for i in range(len(s), 0, -1):
    if s[:i] == rev[len(s) - i:]:
        return rev[:len(s) - i] + s
return s
```

## 25. Checklist trước khi submit

- [ ] Mình đã quy về **tiền tố palindrome dài nhất `L`** và đáp án là `rev[:n - L] + s` (thêm vào **trước**, mảnh dài đúng `n - L`)?
- [ ] Chuỗi nối có **dấu ngăn cách** `'#'` (không xuất hiện trong `s`) — `s + "#" + rev`, không phải `s + rev`?
- [ ] Tính `lps` trên **chuỗi nối**, không phải trên riêng `s`?
- [ ] Lùi biên bằng **`while`** (không phải `if`), có điều kiện `j > 0`, và lùi về `lps[j - 1]` (không phải `lps[j]`)?
- [ ] `s = ""` trả về `""` mà không lỗi?
- [ ] Nếu dùng Cách 2: đã khử đệ quy để tránh `RecursionError` với `n = 5 * 10^4`?
- [ ] Nếu dùng hash: có kiểm tra lại ứng viên để tránh va chạm?
- [ ] Đã thử các ca: `"aacecaaa"`, `"abcd"`, `""`, `"a"`, `"aaba"` (bẫy quên `'#'`), `"aabababa"` (ca xấu Cách 2), chuỗi toàn `'a'` dài 50 000?
- [ ] Độ phức tạp `O(n)` thời gian, `O(n)` bộ nhớ — an toàn với `n = 5 * 10^4`.